# Projet S502 – Migration SQL ⇄ NoSQL par abstraction

BUT3 Sciences des données – S502  
Migration des données SQL ⇄ NoSQL

## Objectifs pédagogiques

L’objectif de ce projet est de démontrer que l’on peut concevoir et manipuler des données
indépendamment de leur mode de persistance, qu’il soit relationnel (SQL) ou non relationnel (NoSQL).

Pour cela, ce projet met en œuvre :
- la normalisation et la dénormalisation des données,
- une représentation abstraite des données via la programmation orientée objet,
- l’utilisation de DAO (Data Access Object) et/ou d’ORM,
- l’automatisation de la migration entre différentes représentations.

L’idée centrale est de rendre la migration SQL ⇄ NoSQL quasi invisible pour le reste de l’application,
en séparant clairement la logique métier de la couche de persistance.


## Donnée brute du projet

Le point de départ du projet est un fichier CSV contenant des données
issues d’un système de mobilité urbaine (Smart City) de la ville de Paris.

Chaque ligne du fichier représente un événement complet,
regroupant des informations relatives :
- à l’utilisateur,
- au véhicule,
- au trajet,
- à l’environnement,
- au paiement,
- ainsi qu’au contexte de risque.

Ce fichier constitue une source brute,
utilisée comme base pour les différentes modélisations
et migrations étudiées dans la suite du projet.



## Exploration de la donnée brute

La première étape consiste à charger et explorer le fichier CSV afin de comprendre
la structure réelle des données avant toute transformation.

Cette exploration permet :
- d’identifier les attributs disponibles,
- de comprendre leur signification métier,
- de faire émerger les futures entités (tables, documents, classes).


In [2]:
import pandas as pd

df = pd.read_csv("donnees_smartcity.csv")
df.shape


(1500, 23)

In [3]:
df.head()

,id_evenement,date_heure,ville,zone,id_utilisateur,id_appareil,type_vehicule,energie,station_depart,station_arrivee,...,pluie,pm10,volume_trafic,montant_paiement,moyen_paiement,nom_utilisateur,age,abonnement,genre,risque_accident
0,1,2025-10-01 00:00,Paris,Sud,192,APP-3143,Velo,Electrique,4,56,...,0,16,148,3.93,Mobile,Lucas,29,Premium,homme,2
1,2,2025-10-01 00:20,Paris,Sud,142,APP-1823,Trottinette,Humaine,33,43,...,1,22,174,1.78,Mobile,Nina,29,Premium,femme,3
2,3,2025-10-01 00:40,Paris,Nord,48,APP-3836,Velo,Humaine,38,35,...,1,19,188,5.31,Carte,Lucas,41,Premium,homme,1
3,4,2025-10-01 01:00,Paris,Est,247,APP-9612,Bus,Electrique,48,34,...,1,21,141,1.35,Carte,Hugo,26,Standard,femme,4
4,5,2025-10-01 01:20,Paris,Est,149,APP-4076,Bus,Humaine,60,24,...,0,16,137,3.05,Mobile,Emma,63,Standard,homme,2


## Structure du jeu de données

Le fichier CSV contient **1500 lignes**, chacune représentant un événement unique,
et **23 colonnes**, décrivant différents aspects de cet événement.


In [4]:
df.shape, df.columns


((1500, 23),
 Index(['id_evenement', 'date_heure', 'ville', 'zone', 'id_utilisateur',
        'id_appareil', 'type_vehicule', 'energie', 'station_depart',
        'station_arrivee', 'distance_km', 'duree_min', 'temperature', 'pluie',
        'pm10', 'volume_trafic', 'montant_paiement', 'moyen_paiement',
        'nom_utilisateur', 'age', 'abonnement', 'genre', 'risque_accident'],
       dtype='object'))

In [5]:
df["ville"].value_counts().head()

ville
Paris    1500
Name: count, dtype: int64

L’analyse des colonnes permet d’identifier plusieurs catégories
d’informations :

- des informations utilisateur (identité, âge, genre, abonnement),
- des informations de trajet (stations, distance, durée),
- des informations environnementales (météo, pollution, trafic),
- des informations de paiement,
- des informations techniques (identifiant d’appareil),
- un indicateur de risque associé à chaque événement.

Cette diversité justifie une structuration plus rigoureuse
des données dans un modèle relationnel.


Le format CSV est simple à manipuler mais présente plusieurs limites :

- les données sont fortement redondantes,
- aucune relation explicite n’existe entre les entités,
- la cohérence des informations n’est pas garantie,
- le format n’est pas adapté à des évolutions du modèle.

Ces limites justifient la mise en place d’un modèle relationnel
normalisé dans la partie suivante.


## Modèle SQL normalisé (référence)

L’objectif de cette partie est de transformer les données brutes du fichier CSV
en un modèle relationnel SQL normalisé.

Ce modèle vise à :
- réduire la redondance des données,
- clarifier les relations entre les entités,
- garantir la cohérence des informations,
- servir de base de référence pour les migrations,
  les dénormalisations et les comparaisons ultérieures.


### Schéma relationnel logique (modèle normalisé)
```
event
 ├─ event_id (PK)
 ├─ datetime
 ├─ zone
 ├─ risque_accident
 ├─ distance_km
 ├─ duration_min
 ├─ city_id (FK)
 ├─ user_id (FK)
 ├─ device_id (FK)
 ├─ vehicle_id (FK)
 ├─ station_start_id (FK)
 ├─ station_end_id (FK)
 ├─ environment_id (FK)
 └─ payment_id (FK)

city
 └─ city_id (PK), name

user
 └─ user_id (PK), name, age, genre, subscription

device
 └─ device_id (PK)

vehicle
 └─ vehicle_id (PK), type, energy

station
 └─ station_id (PK), name, city_id (FK)

environment
 └─ environment_id (PK), temperature, rain, pm10, traffic_volume

payment
 └─ payment_id (PK), amount, method


In [7]:
import sqlite3
import os

DB_SQL_NORMALIZED = "smartcity_sql_normalized.db"

if os.path.exists(DB_SQL_NORMALIZED):
    os.remove(DB_SQL_NORMALIZED)

conn = sqlite3.connect(DB_SQL_NORMALIZED)
cursor = conn.cursor()


In [8]:
cursor.execute("""
CREATE TABLE city (
    city_id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL
)
""")


In [9]:
cursor.execute("""
CREATE TABLE user (
    user_id INTEGER PRIMARY KEY,
    name TEXT,
    age INTEGER,
    genre TEXT,
    subscription TEXT
)
""")


In [10]:
cursor.execute("""
CREATE TABLE device (
    device_id TEXT PRIMARY KEY
)
""")


In [11]:
cursor.execute("""
CREATE TABLE vehicle (
    vehicle_id INTEGER PRIMARY KEY AUTOINCREMENT,
    type TEXT,
    energy TEXT
)
""")


In [12]:
cursor.execute("""
CREATE TABLE station (
    station_id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT,
    city_id INTEGER,
    FOREIGN KEY (city_id) REFERENCES city(city_id)
)
""")


In [13]:
cursor.execute("""
CREATE TABLE environment (
    environment_id INTEGER PRIMARY KEY AUTOINCREMENT,
    temperature REAL,
    rain BOOLEAN,
    pm10 REAL,
    traffic_volume INTEGER
)
""")


In [14]:
cursor.execute("""
CREATE TABLE payment (
    payment_id INTEGER PRIMARY KEY AUTOINCREMENT,
    amount REAL,
    method TEXT
)
""")


In [15]:
cursor.execute("""
CREATE TABLE event (
    event_id INTEGER PRIMARY KEY,
    datetime TEXT,
    zone TEXT,
    risque_accident TEXT,
    city_id INTEGER,
    user_id INTEGER,
    device_id TEXT,
    vehicle_id INTEGER,
    station_start_id INTEGER,
    station_end_id INTEGER,
    environment_id INTEGER,
    payment_id INTEGER,
    distance_km REAL,
    duration_min REAL,
    FOREIGN KEY (city_id) REFERENCES city(city_id),
    FOREIGN KEY (user_id) REFERENCES user(user_id),
    FOREIGN KEY (device_id) REFERENCES device(device_id),
    FOREIGN KEY (vehicle_id) REFERENCES vehicle(vehicle_id),
    FOREIGN KEY (station_start_id) REFERENCES station(station_id),
    FOREIGN KEY (station_end_id) REFERENCES station(station_id),
    FOREIGN KEY (environment_id) REFERENCES environment(environment_id),
    FOREIGN KEY (payment_id) REFERENCES payment(payment_id)
)
""")


La normalisation permet de limiter les redondances
et de garantir la cohérence des données.

Par exemple :
- un utilisateur est stocké une seule fois,
- une ville peut être référencée par plusieurs stations et événements,
- l’appareil est isolé car il s’agit d’un élément technique,
- le risque d’accident dépend du contexte de chaque événement.

Cette structuration facilite les migrations vers d’autres modèles
et la comparaison entre différentes stratégies de stockage.


In [16]:
cursor.execute("PRAGMA table_info(event)").fetchall()

[(0, 'event_id', 'INTEGER', 0, None, 1),
 (1, 'datetime', 'TEXT', 0, None, 0),
 (2, 'zone', 'TEXT', 0, None, 0),
 (3, 'risque_accident', 'TEXT', 0, None, 0),
 (4, 'city_id', 'INTEGER', 0, None, 0),
 (5, 'user_id', 'INTEGER', 0, None, 0),
 (6, 'device_id', 'TEXT', 0, None, 0),
 (7, 'vehicle_id', 'INTEGER', 0, None, 0),
 (8, 'station_start_id', 'INTEGER', 0, None, 0),
 (9, 'station_end_id', 'INTEGER', 0, None, 0),
 (10, 'environment_id', 'INTEGER', 0, None, 0),
 (11, 'payment_id', 'INTEGER', 0, None, 0),
 (12, 'distance_km', 'REAL', 0, None, 0),
 (13, 'duration_min', 'REAL', 0, None, 0)]

In [17]:
cursor.execute("PRAGMA table_info(user)").fetchall()


[(0, 'user_id', 'INTEGER', 0, None, 1),
 (1, 'name', 'TEXT', 0, None, 0),
 (2, 'age', 'INTEGER', 0, None, 0),
 (3, 'genre', 'TEXT', 0, None, 0),
 (4, 'subscription', 'TEXT', 0, None, 0)]

In [18]:
cursor.execute("PRAGMA table_info(device)").fetchall()


[(0, 'device_id', 'TEXT', 0, None, 1)]

In [19]:
conn.commit()
conn.close()


## Insertion des données dans le modèle SQL normalisé

Cette étape consiste à insérer les données issues du fichier CSV
dans la base SQL normalisée définie précédemment.

L’objectif est de :
- peupler la base de référence,
- respecter les relations entre les entités,
- garantir la cohérence des clés primaires et étrangères,
- préparer la base pour les migrations ultérieures.


In [20]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(DB_SQL_NORMALIZED)
cursor = conn.cursor()

df = pd.read_csv("donnees_smartcity.csv")


In [21]:
city_ids = {}

for city in df["ville"].unique():
    cursor.execute(
        "INSERT INTO city (name) VALUES (?)",
        (city,)
    )
    city_ids[city] = cursor.lastrowid


Chaque ville est stockée une seule fois afin d’éviter toute redondance.


In [22]:
device_ids = set()

for device_id in df["id_appareil"].unique():
    cursor.execute(
        "INSERT INTO device (device_id) VALUES (?)",
        (device_id,)
    )
    device_ids.add(device_id)


In [23]:
for _, row in df.iterrows():
    cursor.execute("""
        INSERT OR IGNORE INTO user (
            user_id, name, age, genre, subscription
        )
        VALUES (?, ?, ?, ?, ?)
    """, (
        row["id_utilisateur"],
        row["nom_utilisateur"],
        row["age"],
        row["genre"],
        row["abonnement"]
    ))


L’instruction `INSERT OR IGNORE` évite la duplication des utilisateurs.


In [24]:
vehicle_ids = {}

for _, row in df.iterrows():
    key = (row["type_vehicule"], row["energie"])
    if key not in vehicle_ids:
        cursor.execute("""
            INSERT INTO vehicle (type, energy)
            VALUES (?, ?)
        """, key)
        vehicle_ids[key] = cursor.lastrowid


In [25]:
station_ids = {}

for _, row in df.iterrows():
    for station_name in (row["station_depart"], row["station_arrivee"]):
        key = (station_name, row["ville"])
        if key not in station_ids:
            cursor.execute("""
                INSERT INTO station (name, city_id)
                VALUES (?, ?)
            """, (
                station_name,
                city_ids[row["ville"]]
            ))
            station_ids[key] = cursor.lastrowid


In [26]:
for _, row in df.iterrows():

    # --- Environment ---
    cursor.execute("""
        INSERT INTO environment (
            temperature, rain, pm10, traffic_volume
        )
        VALUES (?, ?, ?, ?)
    """, (
        row["temperature"],
        row["pluie"],
        row["pm10"],
        row["volume_trafic"]
    ))
    env_id = cursor.lastrowid

    # --- Payment ---
    cursor.execute("""
        INSERT INTO payment (amount, method)
        VALUES (?, ?)
    """, (
        row["montant_paiement"],
        row["moyen_paiement"]
    ))
    payment_id = cursor.lastrowid

    # --- Event ---
    cursor.execute("""
        INSERT INTO event (
            event_id, datetime, zone, risque_accident,
            city_id, user_id, device_id, vehicle_id,
            station_start_id, station_end_id,
            environment_id, payment_id,
            distance_km, duration_min
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        row["id_evenement"],
        row["date_heure"],
        row["zone"],
        row["risque_accident"],
        city_ids[row["ville"]],
        row["id_utilisateur"],
        row["id_appareil"],
        vehicle_ids[(row["type_vehicule"], row["energie"])],
        station_ids[(row["station_depart"], row["ville"])],
        station_ids[(row["station_arrivee"], row["ville"])],
        env_id,
        payment_id,
        row["distance_km"],
        row["duree_min"]
    ))


In [27]:
conn.commit()


In [28]:
cursor.execute("SELECT COUNT(*) FROM event").fetchone()


(1500,)

In [29]:
rows = cursor.execute("""
SELECT
    e.event_id,
    e.datetime,
    c.name AS city,
    u.name AS user,
    e.risque_accident,
    v.type AS vehicle
FROM event e
JOIN city c ON e.city_id = c.city_id
JOIN user u ON e.user_id = u.user_id
JOIN vehicle v ON e.vehicle_id = v.vehicle_id
LIMIT 5
""").fetchall()

for row in rows:
    print(row)


(1, '2025-10-01 00:00', 'Paris', 'Lucas', '2', 'Velo')
(2, '2025-10-01 00:20', 'Paris', 'Nina', '3', 'Trottinette')
(3, '2025-10-01 00:40', 'Paris', 'Lucas', '1', 'Velo')
(4, '2025-10-01 01:00', 'Paris', 'Hugo', '4', 'Bus')
(5, '2025-10-01 01:20', 'Paris', 'Emma', '2', 'Bus')


In [30]:
conn.close()


## Deuxième normalisation SQL (variante)

Cette partie propose une seconde modélisation SQL relationnelle
à partir des mêmes données brutes.

L’objectif n’est pas de remplacer le modèle de référence,
mais de montrer qu’il existe plusieurs choix de normalisation possibles,
chacun répondant à des contraintes et usages différents.


### Schéma relationnel logique — Variante
```
event
 ├─ event_id (PK)
 ├─ datetime
 ├─ zone
 ├─ risque_accident
 ├─ temperature
 ├─ rain
 ├─ pm10
 ├─ traffic_volume
 ├─ distance_km
 ├─ duration_min
 ├─ city_id (FK)
 ├─ user_id (FK)
 ├─ device_id (FK)
 ├─ vehicle_id (FK)
 ├─ station_start_id (FK)
 └─ station_end_id (FK)

city
 └─ city_id (PK), name

user
 └─ user_id (PK), name, age, genre, subscription

device
 └─ device_id (PK)

vehicle
 └─ vehicle_id (PK), type, energy

station
 └─ station_id (PK), name, city_id (FK)


Dans cette variante, les informations environnementales
sont directement intégrées à la table événement.

Ce choix permet :
- de réduire le nombre de jointures,
- de simplifier les requêtes de lecture,
- au prix d’une légère redondance des données.

Cette approche est souvent utilisée lorsque les performances
en lecture sont prioritaires par rapport à la normalisation stricte.


In [31]:
DB_SQL_VARIANT = "smartcity_sql_normalized_v2.db"

import sqlite3, os

if os.path.exists(DB_SQL_VARIANT):
    os.remove(DB_SQL_VARIANT)

conn = sqlite3.connect(DB_SQL_VARIANT)
cursor = conn.cursor()


In [32]:
# City
cursor.execute("""
CREATE TABLE city (
    city_id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL
)
""")

# User
cursor.execute("""
CREATE TABLE user (
    user_id INTEGER PRIMARY KEY,
    name TEXT,
    age INTEGER,
    genre TEXT,
    subscription TEXT
)
""")

# Device
cursor.execute("""
CREATE TABLE device (
    device_id TEXT PRIMARY KEY
)
""")

# Vehicle
cursor.execute("""
CREATE TABLE vehicle (
    vehicle_id INTEGER PRIMARY KEY AUTOINCREMENT,
    type TEXT,
    energy TEXT
)
""")

# Station
cursor.execute("""
CREATE TABLE station (
    station_id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT,
    city_id INTEGER,
    FOREIGN KEY (city_id) REFERENCES city(city_id)
)
""")

# Event (VARIANTE)
cursor.execute("""
CREATE TABLE event (
    event_id INTEGER PRIMARY KEY,
    datetime TEXT,
    zone TEXT,
    risque_accident TEXT,
    temperature REAL,
    rain BOOLEAN,
    pm10 REAL,
    traffic_volume INTEGER,
    distance_km REAL,
    duration_min REAL,
    city_id INTEGER,
    user_id INTEGER,
    device_id TEXT,
    vehicle_id INTEGER,
    station_start_id INTEGER,
    station_end_id INTEGER,
    FOREIGN KEY (city_id) REFERENCES city(city_id),
    FOREIGN KEY (user_id) REFERENCES user(user_id),
    FOREIGN KEY (device_id) REFERENCES device(device_id),
    FOREIGN KEY (vehicle_id) REFERENCES vehicle(vehicle_id),
    FOREIGN KEY (station_start_id) REFERENCES station(station_id),
    FOREIGN KEY (station_end_id) REFERENCES station(station_id)
)
""")


Pour des raisons de lisibilité, la logique d’insertion est identique
à celle du modèle de référence, à l’exception des données environnementales
désormais intégrées directement à la table événement.


Cette deuxième normalisation montre qu’il n’existe pas
un unique modèle relationnel correct,
mais des compromis différents selon les usages.


## Abstraction métier (POO)

Cette partie introduit une abstraction orientée objet
représentant le domaine métier de l’application.

Les objets métier constituent le modèle logique central du projet.
Ils sont indépendants :
- du modèle relationnel,
- du modèle NoSQL,
- et des choix de persistance.

Cette abstraction permet de manipuler les données
sans dépendre de leur mode de stockage.


Les classes métier ne correspondent pas directement aux tables SQL.

Une même classe métier peut être persistée :
- dans plusieurs schémas SQL différents,
- ou sous forme de documents NoSQL.

Ce découplage est essentiel pour permettre les migrations
et les comparaisons de modèles.


In [33]:
class City:
    def __init__(self, name):
        self.name = name


In [34]:
class User:
    def __init__(self, user_id, name, age, genre, subscription):
        self.user_id = user_id
        self.name = name
        self.age = age
        self.genre = genre
        self.subscription = subscription


In [35]:
class Device:
    def __init__(self, device_id):
        self.device_id = device_id


In [36]:
class Vehicle:
    def __init__(self, vehicle_type, energy):
        self.vehicle_type = vehicle_type
        self.energy = energy


In [37]:
class Station:
    def __init__(self, name, city):
        self.name = name
        self.city = city


In [38]:
class Environment:
    def __init__(self, temperature, rain, pm10, traffic_volume):
        self.temperature = temperature
        self.rain = rain
        self.pm10 = pm10
        self.traffic_volume = traffic_volume


In [39]:
class Payment:
    def __init__(self, amount, method):
        self.amount = amount
        self.method = method


In [40]:
class Event:
    def __init__(
        self,
        event_id,
        datetime,
        zone,
        risque_accident,
        city,
        user,
        device,
        vehicle,
        station_start,
        station_end,
        distance_km,
        duration_min,
        environment=None,
        payment=None
    ):
        self.event_id = event_id
        self.datetime = datetime
        self.zone = zone
        self.risque_accident = risque_accident
        self.city = city
        self.user = user
        self.device = device
        self.vehicle = vehicle
        self.station_start = station_start
        self.station_end = station_end
        self.distance_km = distance_km
        self.duration_min = duration_min
        self.environment = environment
        self.payment = payment


```
Event
 ├─ City
 ├─ User
 ├─ Device
 ├─ Vehicle
 ├─ Station (start)
 ├─ Station (end)
 ├─ Environment
 └─ Payment


L’abstraction métier permet de représenter les données
indépendamment des choix de stockage, ce qui rend les migrations possibles
sans modifier la logique métier.

## DAO SQL

Les DAO (Data Access Object) constituent la couche chargée
d’accéder aux données stockées en base SQL.

Ils assurent la séparation entre :
- la logique métier (objets),
- la logique de persistance (SQL).

Grâce aux DAO, les objets métier peuvent être reconstruits
à partir de différentes bases SQL sans modifier leur définition.


Le DAO SQL est responsable de :
- l’ouverture de la connexion à la base,
- l’exécution des requêtes SQL (jointures),
- la reconstruction des objets métier.

Aucune logique métier n’est présente dans le DAO.


In [41]:
import sqlite3

class EventSQLDao:
    def __init__(self, db_path):
        self.db_path = db_path


In [42]:
class EventSQLDao:
    def __init__(self, db_path):
        self.db_path = db_path

    def fetch_all(self):
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()

        query = """
        SELECT
            e.event_id,
            e.datetime,
            e.zone,
            e.risque_accident,
            c.name AS city_name,
            u.user_id, u.name, u.age, u.genre, u.subscription,
            d.device_id,
            v.type, v.energy,
            s1.name AS station_start,
            s2.name AS station_end,
            e.distance_km,
            e.duration_min,
            env.temperature, env.rain, env.pm10, env.traffic_volume,
            p.amount, p.method
        FROM event e
        JOIN city c ON e.city_id = c.city_id
        JOIN user u ON e.user_id = u.user_id
        JOIN device d ON e.device_id = d.device_id
        JOIN vehicle v ON e.vehicle_id = v.vehicle_id
        JOIN station s1 ON e.station_start_id = s1.station_id
        JOIN station s2 ON e.station_end_id = s2.station_id
        JOIN environment env ON e.environment_id = env.environment_id
        JOIN payment p ON e.payment_id = p.payment_id
        """

        rows = cursor.execute(query).fetchall()
        conn.close()

        events = []

        for row in rows:
            city = City(row[4])
            user = User(row[5], row[6], row[7], row[8], row[9])
            device = Device(row[10])
            vehicle = Vehicle(row[11], row[12])
            station_start = Station(row[13], city)
            station_end = Station(row[14], city)
            environment = Environment(row[17], row[18], row[19], row[20])
            payment = Payment(row[21], row[22])

            event = Event(
                event_id=row[0],
                datetime=row[1],
                zone=row[2],
                risque_accident=row[3],
                city=city,
                user=user,
                device=device,
                vehicle=vehicle,
                station_start=station_start,
                station_end=station_end,
                distance_km=row[15],
                duration_min=row[16],
                environment=environment,
                payment=payment
            )

            events.append(event)

        return events


In [43]:
sql_dao = EventSQLDao(DB_SQL_NORMALIZED)
events = sql_dao.fetch_all()

len(events)


1500

In [44]:
events[0].__dict__

{'event_id': 1,
 'datetime': '2025-10-01 00:00',
 'zone': 'Sud',
 'risque_accident': '2',
 'city': <__main__.City at 0x1e278cc4ec0>,
 'user': <__main__.User at 0x1e278cc5400>,
 'device': <__main__.Device at 0x1e278cc5550>,
 'vehicle': <__main__.Vehicle at 0x1e278cc5940>,
 'station_start': <__main__.Station at 0x1e278cc5a90>,
 'station_end': <__main__.Station at 0x1e278caec10>,
 'distance_km': 4.66,
 'duration_min': 3.0,
 'environment': <__main__.Environment at 0x1e278cc5be0>,
 'payment': <__main__.Payment at 0x1e278cc5d30>}

Le DAO SQL transforme les données relationnelles
en objets métier sans exposer la structure de la base.

## MongoDB (base documentaire de référence)

Cette partie introduit MongoDB comme système de gestion de données NoSQL
orienté document, à l’aide de la bibliothèque PyMongo.

Cette première modélisation NoSQL n’a pas pour objectif la dénormalisation,
mais sert de base de référence avant l’exploration de plusieurs stratégies
de dénormalisation dans les parties suivantes.


Le modèle documentaire de référence reprend la structure logique
du modèle relationnel normalisé.

Chaque document représente un événement,
avec des sous-documents pour les entités associées,
sans choix de dénormalisation spécifique orientée usage.


### Structure logique du document MongoDB (référence)

```
{
  event_id,
  datetime,
  zone,
  risque_accident,
  city_id,
  user_id,
  device_id,
  vehicle_id,
  station_start_id,
  station_end_id,
  environment_id,
  payment_id,
  distance_km,
  duration_min
}


In [45]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/")
db = client["smartcity"]
collection_ref = db["events_reference"]


MongoDB doit être lancé localement pour exécuter cette partie.

In [46]:
collection_ref.delete_many({})
# Nettoyage de la collection (rejouable)

DeleteResult({'n': 0, 'ok': 1.0}, acknowledged=True)

In [47]:
class EventMongoReferenceDao:
    def __init__(self, collection):
        self.collection = collection
# DAO MongoDB (référence)
#DAO minimal, sans dénormalisation.

In [48]:
class EventMongoReferenceDao:
    def __init__(self, collection):
        self.collection = collection

    def insert_many(self, events):
        docs = []

        for event in events:
            doc = {
                "event_id": event.event_id,
                "datetime": event.datetime,
                "zone": event.zone,
                "risque_accident": event.risque_accident,
                "city_id": event.city.name,
                "user_id": event.user.user_id,
                "device_id": event.device.device_id,
                "vehicle": {
                    "type": event.vehicle.vehicle_type,
                    "energy": event.vehicle.energy
                },
                "stations": {
                    "start": event.station_start.name,
                    "end": event.station_end.name
                },
                "distance_km": event.distance_km,
                "duration_min": event.duration_min
            }

            docs.append(doc)

        if docs:
            self.collection.insert_many(docs)

# Insertion depuis les objets métier


In [49]:
sql_dao = EventSQLDao(DB_SQL_NORMALIZED)
events = sql_dao.fetch_all()

mongo_ref_dao = EventMongoReferenceDao(collection_ref)
mongo_ref_dao.insert_many(events)
# Migration SQL → MongoDB (référence)

In [50]:
collection_ref.count_documents({})
# Validation de la base MongoDB de référence

1500

In [51]:
collection_ref.find_one()

{'_id': ObjectId('697e298761deb6479274eac8'),
 'event_id': 1,
 'datetime': '2025-10-01 00:00',
 'zone': 'Sud',
 'risque_accident': '2',
 'city_id': 'Paris',
 'user_id': 192,
 'device_id': 'APP-3143',
 'vehicle': {'type': 'Velo', 'energy': 'Electrique'},
 'stations': {'start': '4', 'end': '56'},
 'distance_km': 4.66,
 'duration_min': 3.0}

## DÉNORMALISATION N°1 — EVENT DÉTAILLÉ
Je suis un opérateur du service Smart City ou un agent de support.

Mon rôle est de consulter rapidement le détail complet d’un trajet
lorsqu’un utilisateur me contacte ou lorsqu’un incident est signalé.

Je dois accéder immédiatement à toutes les informations liées à un
événement précis (trajet, utilisateur, véhicule, environnement,
paiement), sans avoir à naviguer entre plusieurs écrans ni à exécuter
des requêtes SQL complexes.

## OBJECTIF DATA

L’objectif de cette dénormalisation est de regrouper dans un document
unique toutes les informations nécessaires à la consultation complète
d’un événement.

Ce modèle vise à :
- minimiser le temps de réponse,
- éviter les jointures,
- simplifier l’accès aux données opérationnelles.


## COLONNES UTILES (SOURCES SQL)

À partir de la table `user` :
- `id_utilisateur` : identifiant utilisateur
- `nom_utilisateur` : identification humaine
- `age` : information contextuelle
- `genre` : information contextuelle
- `abonnement` : type de service utilisé

À partir de la table `city` :
- `ville` : localisation de l’événement

À partir de la table `user` :
- `id_utilisateur` : identifiant utilisateur
- `nom_utilisateur` : identification humaine
- `age` : information contextuelle
- `genre` : information contextuelle
- `abonnement` : type de service utilisé

À partir de la table `vehicle` :
- `type_vehicule` : type de transport utilisé
- `energie` : motorisation

À partir de la table `station` :
- `station_depart`
- `station_arrivee`

À partir de la table `environment` :
- `temperature`
- `pluie`
- `pm10`
- `volume_trafic`

À partir de la table `payment` :
- `montant_paiement`
- `moyen_paiement`


In [52]:
# Schéma logique du document event_detaillé

{
  "event_id": 845,
  "date_heure": "2025-10-28 14:23",
  "zone": "Centre",
  "ville": "Paris",
  "risque_accident": 3,

  "trajet": {
    "station_depart": "Gare de Lyon",
    "station_arrivee": "Bastille",
    "distance_km": 4.2,
    "duree_min": 12.5
  },

  "utilisateur": {
    "id_utilisateur": 192,
    "nom_utilisateur": "Lucas",
    "age": 29,
    "genre": "homme",
    "abonnement": "Premium"
  },

  "vehicule": {
    "type": "Velo",
    "energie": "Electrique"
  },

  "environnement": {
    "temperature": 18.2,
    "pluie": False,
    "pm10": 21.4,
    "volume_trafic": 1450
  },

  "paiement": {
    "montant": 3.90,
    "moyen": "Mobile"
  },

  "derniere_mise_a_jour": "2026-01-28T02:00:00"
}


{'event_id': 845,
 'date_heure': '2025-10-28 14:23',
 'zone': 'Centre',
 'ville': 'Paris',
 'risque_accident': 3,
 'trajet': {'station_depart': 'Gare de Lyon',
  'station_arrivee': 'Bastille',
  'distance_km': 4.2,
  'duree_min': 12.5},
 'utilisateur': {'id_utilisateur': 192,
  'nom_utilisateur': 'Lucas',
  'age': 29,
  'genre': 'homme',
  'abonnement': 'Premium'},
 'vehicule': {'type': 'Velo', 'energie': 'Electrique'},
 'environnement': {'temperature': 18.2,
  'pluie': False,
  'pm10': 21.4,
  'volume_trafic': 1450},
 'paiement': {'montant': 3.9, 'moyen': 'Mobile'},
 'derniere_mise_a_jour': '2026-01-28T02:00:00'}

#### JUSTIFICATION TECHNIQUE — Performance
Dans le modèle relationnel normalisé, la consultation complète
d’un événement nécessite plusieurs jointures entre les tables :
event, user, city, vehicle, station, environment et payment.

Ces jointures ont un coût non négligeable,
en particulier lorsque les consultations sont fréquentes.

La duplication des informations utilisateur,
véhicule et environnement est volontaire.

Elle est acceptable car :
- les données sont majoritairement en lecture,
- les mises à jour sont rares,
- la cohérence forte en temps réel n’est pas requise.



##### IMPLÉMENTATION PyMongo

In [53]:
from datetime import datetime
collection_event_detail = db["events_event_detail"]
# Création de la collection dédiée

In [54]:
collection_event_detail.delete_many({})
# Nettoyage de la collection (rejouable)


DeleteResult({'n': 0, 'ok': 1.0}, acknowledged=True)

In [55]:
class EventMongoEventDetailDao:
    def __init__(self, collection):
        self.collection = collection
# DAO MongoDB (détaillé)

In [56]:
class EventMongoEventDetailDao:
    def __init__(self, collection):
        self.collection = collection

    def insert_many(self, events):
        documents = []

        for event in events:
            doc = {
                "event_id": event.event_id,
                "date_heure": event.datetime,
                "zone": event.zone,
                "ville": event.city.name,
                "risque_accident": event.risque_accident,

                "trajet": {
                    "station_depart": event.station_start.name,
                    "station_arrivee": event.station_end.name,
                    "distance_km": event.distance_km,
                    "duree_min": event.duration_min
                },

                "utilisateur": {
                    "id_utilisateur": event.user.user_id,
                    "nom_utilisateur": event.user.name,
                    "age": event.user.age,
                    "genre": event.user.genre,
                    "abonnement": event.user.subscription
                },

                "vehicule": {
                    "type": event.vehicle.vehicle_type,
                    "energie": event.vehicle.energy
                },

                "environnement": {
                    "temperature": event.environment.temperature,
                    "pluie": event.environment.rain,
                    "pm10": event.environment.pm10,
                    "volume_trafic": event.environment.traffic_volume
                },

                "paiement": {
                    "montant": event.payment.amount,
                    "moyen": event.payment.method
                },

                "derniere_mise_a_jour": datetime.utcnow().isoformat()
            }

            documents.append(doc)

        if documents:
            self.collection.insert_many(documents)


In [57]:
mongo_event_detail_dao = EventMongoEventDetailDao(collection_event_detail)
mongo_event_detail_dao.insert_many(events)
# Migration SQL → MongoDB (détaillé)

C:\Users\Pc\AppData\Local\Temp\ipykernel_24908\2584089373.py:48: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "derniere_mise_a_jour": datetime.utcnow().isoformat()


In [58]:
collection_event_detail.count_documents({})
# Validation de la base MongoDB détaillée

1500

In [59]:
collection_event_detail.find_one()


{'_id': ObjectId('697e298861deb6479274f0a4'),
 'event_id': 1,
 'date_heure': '2025-10-01 00:00',
 'zone': 'Sud',
 'ville': 'Paris',
 'risque_accident': '2',
 'trajet': {'station_depart': '4',
  'station_arrivee': '56',
  'distance_km': 4.66,
  'duree_min': 3.0},
 'utilisateur': {'id_utilisateur': 192,
  'nom_utilisateur': 'Lucas',
  'age': 29,
  'genre': 'homme',
  'abonnement': 'Premium'},
 'vehicule': {'type': 'Velo', 'energie': 'Electrique'},
 'environnement': {'temperature': 2.0,
  'pluie': 0,
  'pm10': 16.0,
  'volume_trafic': 148},
 'paiement': {'montant': 3.93, 'moyen': 'Mobile'},
 'derniere_mise_a_jour': '2026-01-31T16:10:48.252365'}

In [60]:
collection_event_detail.find_one({"event_id": 845})

{'_id': ObjectId('697e298861deb6479274f3f0'),
 'event_id': 845,
 'date_heure': '2025-10-12 17:20',
 'zone': 'Sud',
 'ville': 'Paris',
 'risque_accident': '2',
 'trajet': {'station_depart': '14',
  'station_arrivee': '14',
  'distance_km': 7.67,
  'duree_min': 23.0},
 'utilisateur': {'id_utilisateur': 203,
  'nom_utilisateur': 'Alice',
  'age': 25,
  'genre': 'femme',
  'abonnement': 'Gratuit'},
 'vehicule': {'type': 'Bus', 'energie': 'Diesel'},
 'environnement': {'temperature': 1.6,
  'pluie': 0,
  'pm10': 20.0,
  'volume_trafic': 219},
 'paiement': {'montant': 3.72, 'moyen': 'Carte'},
 'derniere_mise_a_jour': '2026-01-31T16:10:48.261039'}

════════════════════════════════════════════════════════════════════════════
               DÉNORMALISATION N°2 — SÉCURITÉ PAR ZONE
════════════════════════════════════════════════════════════════════════════

##### USAGE (persona métier)
Je suis responsable de la sécurité routière d’une grande ville.

Mon équipe et moi analysons quotidiennement les zones à risque
afin de déployer des agents de prévention et lancer des campagnes ciblées.

J’ai besoin d’un accès IMMÉDIAT aux indicateurs clés de chaque zone :
niveau de risque, pollution, conditions météo, type de véhicule dominant,
et volume d’événements, sans écrire de requêtes SQL complexes.


##### OBJECTIF DATA
Fournir un document synthétique par zone géographique
contenant des indicateurs de sécurité consolidés,
prêts à être consultés ou visualisés dans un tableau de bord.


##### COLONNES UTILES (sources SQL)
Table event :
- zone
- event_id
- risque_accident
- distance_km

Table vehicle :
- type_vehicule

Table environment :
- temperature
- pluie
- pm10
- volume_trafic


In [61]:
# FORMAT DOCUMENT (MongoDB)

{
  "zone": "Nord",

  "vehicule_dominant": "Velo",

  "statistiques": {
    "risque_accident_moyen": 2.45,
    "pollution_pm10_moyenne": 19.8,
    "temperature_moyenne": 1.9,
    "frequence_pluie": 0.73,
    "nombre_evenements_total": 287
  },

  "repartition_vehicules": [
    { "type": "Velo", "nombre_evenements": 145, "pourcentage": 50.5 },
    { "type": "Bus", "nombre_evenements": 87, "pourcentage": 30.3 },
    { "type": "Trottinette", "nombre_evenements": 55, "pourcentage": 19.2 }
  ],

  "alerte_securite": "MOYENNE",
  "derniere_mise_a_jour": "2026-01-28T10:30:00"
}


{'zone': 'Nord',
 'vehicule_dominant': 'Velo',
 'statistiques': {'risque_accident_moyen': 2.45,
  'pollution_pm10_moyenne': 19.8,
  'temperature_moyenne': 1.9,
  'frequence_pluie': 0.73,
  'nombre_evenements_total': 287},
 'repartition_vehicules': [{'type': 'Velo',
   'nombre_evenements': 145,
   'pourcentage': 50.5},
  {'type': 'Bus', 'nombre_evenements': 87, 'pourcentage': 30.3},
  {'type': 'Trottinette', 'nombre_evenements': 55, 'pourcentage': 19.2}],
 'alerte_securite': 'MOYENNE',
 'derniere_mise_a_jour': '2026-01-28T10:30:00'}

In [62]:
# IMPLÉMENTATION PyMongo
from collections import Counter
from datetime import datetime

collection_zone = db["zones_securite"]
collection_zone.delete_many({})

zones = {}

for event in events:
    zone = event.zone

    if zone not in zones:
        zones[zone] = {
            "zone": zone,
            "risques": [],
            "pm10": [],
            "temperatures": [],
            "pluies": [],
            "vehicules": []
        }

    zones[zone]["risques"].append(float(event.risque_accident))
    zones[zone]["pm10"].append(float(event.environment.pm10))
    zones[zone]["temperatures"].append(float(event.environment.temperature))
    zones[zone]["pluies"].append(int(event.environment.rain))

    zones[zone]["vehicules"].append(event.vehicle.vehicle_type)

documents = []

for zone, data in zones.items():
    total = len(data["risques"])
    compteur_vehicules = Counter(data["vehicules"])
    vehicule_dominant = compteur_vehicules.most_common(1)[0][0]

    repartition = [
        {
            "type": v,
            "nombre_evenements": n,
            "pourcentage": round(n / total * 100, 1)
        }
        for v, n in compteur_vehicules.items()
    ]

    risque_moyen = sum(data["risques"]) / total
    alerte = "ELEVEE" if risque_moyen > 3 else "MOYENNE" if risque_moyen > 2 else "FAIBLE"

    documents.append({
        "zone": zone,
        "vehicule_dominant": vehicule_dominant,
        "statistiques": {
            "risque_accident_moyen": round(risque_moyen, 2),
            "pollution_pm10_moyenne": round(sum(data["pm10"]) / total, 2),
            "temperature_moyenne": round(sum(data["temperatures"]) / total, 2),
            "frequence_pluie": round(sum(data["pluies"]) / total, 2),
            "nombre_evenements_total": total
        },
        "repartition_vehicules": repartition,
        "alerte_securite": alerte,
        "derniere_mise_a_jour": datetime.utcnow().isoformat()
    })

collection_zone.insert_many(documents)


C:\Users\Pc\AppData\Local\Temp\ipykernel_24908\1067954402.py:61: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "derniere_mise_a_jour": datetime.utcnow().isoformat()


InsertManyResult([ObjectId('697e298961deb6479274f680'), ObjectId('697e298961deb6479274f681'), ObjectId('697e298961deb6479274f682'), ObjectId('697e298961deb6479274f683'), ObjectId('697e298961deb6479274f684')], acknowledged=True)

In [63]:
collection_zone.count_documents({})
# Nombre de zones

5

In [64]:
collection_zone.find_one()

{'_id': ObjectId('697e298961deb6479274f680'),
 'zone': 'Sud',
 'vehicule_dominant': 'Velo',
 'statistiques': {'risque_accident_moyen': 2.59,
  'pollution_pm10_moyenne': 21.35,
  'temperature_moyenne': 2.14,
  'frequence_pluie': 0.59,
  'nombre_evenements_total': 298},
 'repartition_vehicules': [{'type': 'Velo',
   'nombre_evenements': 81,
   'pourcentage': 27.2},
  {'type': 'Trottinette', 'nombre_evenements': 66, 'pourcentage': 22.1},
  {'type': 'Bus', 'nombre_evenements': 73, 'pourcentage': 24.5},
  {'type': 'Voiture', 'nombre_evenements': 78, 'pourcentage': 26.2}],
 'alerte_securite': 'MOYENNE',
 'derniere_mise_a_jour': '2026-01-31T16:10:49.264752'}

════════════════════════════════════════════════════════════════════════════
               DÉNORMALISATION N°3 — PROFIL UTILISATEUR MARKETING
════════════════════════════════════════════════════════════════════════════


##### USAGE (persona métier)
Je suis responsable marketing du service Smart City.

Mon objectif est de comprendre finement les comportements
des utilisateurs afin de personnaliser les offres d’abonnement,
les campagnes promotionnelles et les recommandations.

J’ai besoin d’un accès RAPIDE au profil complet de chaque utilisateur :
activité, préférences de mobilité, dépenses et engagement,
sans exécuter de requêtes SQL complexes.


##### OBJECTIF DATA
Construire un document synthétique par utilisateur
regroupant ses statistiques d’usage, ses préférences
et des indicateurs d’engagement exploitables par les équipes marketing.


##### COLONNES UTILES (sources SQL)
Table user :
- id_utilisateur
- nom_utilisateur
- age
- genre
- abonnement

Table event :
- event_id
- distance_km
- duree_min
- date_heure

Table vehicle :
- type_vehicule

Table payment :
- montant_paiement
- moyen_paiement


In [65]:
# FORMAT DOCUMENT (MongoDB)
{
  "id_utilisateur": 192,
  "nom_utilisateur": "Lucas",
  "age": 29,
  "genre": "homme",
  "abonnement_actuel": "Premium",

  "statistiques_usage": {
    "nombre_trajets": 47,
    "distance_moyenne_km": 5.8,
    "duree_moyenne_min": 14.2,
    "depenses_totales_euros": 187.45,
    "frequence_hebdomadaire": 6.7
  },

  "preferences": {
    "vehicule_prefere": "Velo",
    "moyen_paiement_prefere": "Mobile"
  },

  "score_engagement": 8.5,
  "derniere_activite": "2025-10-30 18:45",
  "derniere_mise_a_jour": "2026-01-28T02:00:00"
}


{'id_utilisateur': 192,
 'nom_utilisateur': 'Lucas',
 'age': 29,
 'genre': 'homme',
 'abonnement_actuel': 'Premium',
 'statistiques_usage': {'nombre_trajets': 47,
  'distance_moyenne_km': 5.8,
  'duree_moyenne_min': 14.2,
  'depenses_totales_euros': 187.45,
  'frequence_hebdomadaire': 6.7},
 'preferences': {'vehicule_prefere': 'Velo',
  'moyen_paiement_prefere': 'Mobile'},
 'score_engagement': 8.5,
 'derniere_activite': '2025-10-30 18:45',
 'derniere_mise_a_jour': '2026-01-28T02:00:00'}

In [66]:
# IMPLÉMENTATION PyMongo — DÉNORMALISATION N°3 (CORRIGÉE)
from collections import Counter
from datetime import datetime
from statistics import mean

collection_users = db["users_marketing"]
collection_users.delete_many({})

users = {}

# =========================
# COLLECTE DES DONNÉES
# =========================
for event in events:
    uid = event.user.user_id

    if uid not in users:
        users[uid] = {
            "id_utilisateur": uid,
            "nom_utilisateur": event.user.name,
            "age": event.user.age,
            "genre": event.user.genre,
            "abonnement_actuel": event.user.subscription,
            "distances": [],
            "durees": [],
            "vehicules": [],
            "montants": [],
            "moyens_paiement": [],
            "dates": []
        }

    users[uid]["distances"].append(float(event.distance_km))
    users[uid]["durees"].append(float(event.duration_min))
    users[uid]["vehicules"].append(event.vehicle.vehicle_type)
    users[uid]["montants"].append(float(event.payment.amount))
    users[uid]["moyens_paiement"].append(event.payment.method)
    users[uid]["dates"].append(event.datetime)

# =========================
# CONSTRUCTION DES DOCUMENTS
# =========================
documents = []

for uid, data in users.items():
    nb = len(data["distances"])

    vehicule_pref = (
        Counter(data["vehicules"]).most_common(1)[0][0]
        if data["vehicules"]
        else "Inconnu"
    )

    moyen_paiement_pref = (
        Counter(data["moyens_paiement"]).most_common(1)[0][0]
        if data["moyens_paiement"]
        else "Inconnu"
    )

    freq_hebdo = round(nb / 4, 1)  # approximation simple

    documents.append({
        "id_utilisateur": uid,
        "nom_utilisateur": data["nom_utilisateur"],
        "age": data["age"],
        "genre": data["genre"],
        "abonnement_actuel": data["abonnement_actuel"],

        "statistiques_usage": {
            "nombre_trajets": nb,
            "distance_moyenne_km": round(mean(data["distances"]), 2) if nb else 0,
            "duree_moyenne_min": round(mean(data["durees"]), 2) if nb else 0,
            "depenses_totales_euros": round(sum(data["montants"]), 2),
            "frequence_hebdomadaire": freq_hebdo
        },

        "preferences": {
            "vehicule_prefere": vehicule_pref,
            "moyen_paiement_prefere": moyen_paiement_pref
        },

        "score_engagement": round(min(10, freq_hebdo * 1.2), 1),
        "derniere_activite": max(data["dates"]),
        "derniere_mise_a_jour": datetime.utcnow().isoformat()
    })

# =========================
# INSERTION MONGODB
# =========================
collection_users.insert_many(documents)


C:\Users\Pc\AppData\Local\Temp\ipykernel_24908\1374920102.py:83: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "derniere_mise_a_jour": datetime.utcnow().isoformat()


InsertManyResult([ObjectId('697e298961deb6479274f685'), ObjectId('697e298961deb6479274f686'), ObjectId('697e298961deb6479274f687'), ObjectId('697e298961deb6479274f688'), ObjectId('697e298961deb6479274f689'), ObjectId('697e298961deb6479274f68a'), ObjectId('697e298961deb6479274f68b'), ObjectId('697e298961deb6479274f68c'), ObjectId('697e298961deb6479274f68d'), ObjectId('697e298961deb6479274f68e'), ObjectId('697e298961deb6479274f68f'), ObjectId('697e298961deb6479274f690'), ObjectId('697e298961deb6479274f691'), ObjectId('697e298961deb6479274f692'), ObjectId('697e298961deb6479274f693'), ObjectId('697e298961deb6479274f694'), ObjectId('697e298961deb6479274f695'), ObjectId('697e298961deb6479274f696'), ObjectId('697e298961deb6479274f697'), ObjectId('697e298961deb6479274f698'), ObjectId('697e298961deb6479274f699'), ObjectId('697e298961deb6479274f69a'), ObjectId('697e298961deb6479274f69b'), ObjectId('697e298961deb6479274f69c'), ObjectId('697e298961deb6479274f69d'), ObjectId('697e298961deb6479274f6

In [67]:
collection_users.count_documents({})

250

In [68]:
collection_users.find_one()

{'_id': ObjectId('697e298961deb6479274f685'),
 'id_utilisateur': 192,
 'nom_utilisateur': 'Lucas',
 'age': 29,
 'genre': 'homme',
 'abonnement_actuel': 'Premium',
 'statistiques_usage': {'nombre_trajets': 5,
  'distance_moyenne_km': 8.05,
  'duree_moyenne_min': 13.6,
  'depenses_totales_euros': 20.17,
  'frequence_hebdomadaire': 1.2},
 'preferences': {'vehicule_prefere': 'Velo',
  'moyen_paiement_prefere': 'Mobile'},
 'score_engagement': 1.4,
 'derniere_activite': '2025-10-17 07:20',
 'derniere_mise_a_jour': '2026-01-31T16:10:49.841295'}

════════════════════════════════════════════════════════════════════════════
          DÉNORMALISATION N°4 — ANALYTICS & PILOTAGE GLOBAL
════════════════════════════════════════════════════════════════════════════


##### USAGE (persona métier)
Je suis directeur de la mobilité urbaine de la métropole.

Mon rôle est de piloter la stratégie globale du service Smart City
à l’aide d’indicateurs synthétiques fiables.

Je consulte régulièrement des tableaux de bord pour suivre :
- l’usage du service,
- la sécurité,
- l’impact environnemental,
- la performance économique.

J’ai besoin d’indicateurs PRÉ-CALCULÉS,
sans lancer de requêtes analytiques lourdes.


##### OBJECTIF DATA
Construire un document analytique consolidé
permettant un pilotage global du service,
avec des KPI directement exploitables
par les équipes de direction et de décision.


##### COLONNES UTILES (sources SQL)
Table event :
- event_id
- date_heure
- distance_km
- duree_min
- risque_accident
- zone
- ville

Table vehicle :
- type_vehicule

Table environment :
- pm10
- volume_trafic
- pluie

Table payment :
- montant_paiement


In [69]:
# FORMAT DOCUMENT (MongoDB)
{
  "periode": "2025-10",

  "kpi_globaux": {
    "nombre_trajets": 1500,
    "distance_totale_km": 8420.5,
    "duree_moyenne_min": 13.7,
    "chiffre_affaires_total": 5123.8,
    "risque_moyen": 2.3,
    "pollution_moyenne_pm10": 21.4
  },

  "repartition_vehicules": [
    { "type": "Velo", "nombre_trajets": 620, "pourcentage": 41.3 },
    { "type": "Bus", "nombre_trajets": 480, "pourcentage": 32.0 },
    { "type": "Trottinette", "nombre_trajets": 400, "pourcentage": 26.7 }
  ],

  "indicateurs_securite": {
    "zones_a_risque": ["Nord", "Est"],
    "taux_pluie": 0.36
  },

  "impact_environnemental": {
    "volume_trafic_moyen": 1450
  },

  "derniere_mise_a_jour": "2026-01-28T02:00:00"
}


{'periode': '2025-10',
 'kpi_globaux': {'nombre_trajets': 1500,
  'distance_totale_km': 8420.5,
  'duree_moyenne_min': 13.7,
  'chiffre_affaires_total': 5123.8,
  'risque_moyen': 2.3,
  'pollution_moyenne_pm10': 21.4},
 'repartition_vehicules': [{'type': 'Velo',
   'nombre_trajets': 620,
   'pourcentage': 41.3},
  {'type': 'Bus', 'nombre_trajets': 480, 'pourcentage': 32.0},
  {'type': 'Trottinette', 'nombre_trajets': 400, 'pourcentage': 26.7}],
 'indicateurs_securite': {'zones_a_risque': ['Nord', 'Est'],
  'taux_pluie': 0.36},
 'impact_environnemental': {'volume_trafic_moyen': 1450},
 'derniere_mise_a_jour': '2026-01-28T02:00:00'}

In [70]:
# IMPLÉMENTATION PyMongo
from collections import Counter
from datetime import datetime

collection_analytics = db["analytics_global"]
collection_analytics.delete_many({})

distances = []
durees = []
risques = []
pm10s = []
pluies = []
paiements = []
vehicules = []
zones_risque = set()

for event in events:
    distances.append(float(event.distance_km))
    durees.append(float(event.duration_min))
    risques.append(float(event.risque_accident))
    pm10s.append(float(event.environment.pm10))
    pluies.append(int(event.environment.rain))
    paiements.append(float(event.payment.amount))
    vehicules.append(event.vehicle.vehicle_type)

    if float(event.risque_accident) > 3:
        zones_risque.add(event.zone)

total = len(distances)
vehicules_count = Counter(vehicules)

document = {
    "periode": "2025-10",
    "kpi_globaux": {
        "nombre_trajets": total,
        "distance_totale_km": round(sum(distances), 2),
        "duree_moyenne_min": round(sum(durees) / total, 2),
        "chiffre_affaires_total": round(sum(paiements), 2),
        "risque_moyen": round(sum(risques) / total, 2),
        "pollution_moyenne_pm10": round(sum(pm10s) / total, 2)
    },
    "repartition_vehicules": [
        {
            "type": v,
            "nombre_trajets": n,
            "pourcentage": round(n / total * 100, 1)
        }
        for v, n in vehicules_count.items()
    ],
    "indicateurs_securite": {
        "zones_a_risque": list(zones_risque),
        "taux_pluie": round(sum(pluies) / total, 2)
    },
    "impact_environnemental": {
        "volume_trafic_moyen": round(sum(pm10s) / total, 2)
    },
    "derniere_mise_a_jour": datetime.utcnow().isoformat()
}

collection_analytics.insert_one(document)


C:\Users\Pc\AppData\Local\Temp\ipykernel_24908\1168073772.py:57: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "derniere_mise_a_jour": datetime.utcnow().isoformat()


InsertOneResult(ObjectId('697e298a61deb6479274f77f'), acknowledged=True)

In [71]:
collection_analytics.find_one()

{'_id': ObjectId('697e298a61deb6479274f77f'),
 'periode': '2025-10',
 'kpi_globaux': {'nombre_trajets': 1500,
  'distance_totale_km': 9314.02,
  'duree_moyenne_min': 13.47,
  'chiffre_affaires_total': 5370.23,
  'risque_moyen': 2.51,
  'pollution_moyenne_pm10': 21.12},
 'repartition_vehicules': [{'type': 'Velo',
   'nombre_trajets': 380,
   'pourcentage': 25.3},
  {'type': 'Trottinette', 'nombre_trajets': 366, 'pourcentage': 24.4},
  {'type': 'Bus', 'nombre_trajets': 373, 'pourcentage': 24.9},
  {'type': 'Voiture', 'nombre_trajets': 381, 'pourcentage': 25.4}],
 'indicateurs_securite': {'zones_a_risque': ['Centre',
   'Sud',
   'Nord',
   'Est',
   'Ouest'],
  'taux_pluie': 0.59},
 'impact_environnemental': {'volume_trafic_moyen': 21.12},
 'derniere_mise_a_jour': '2026-01-31T16:10:50.629590'}

##### MIGRATION INVERSE
##### NoSQL → SQL normalisé

Dans cette partie, nous implémentons une migration inverse
pour chacune des quatre dénormalisations précédemment définies.

Chaque migration inverse produit une normalisation relationnelle
cohérente avec les données disponibles dans la dénormalisation source.

L’objectif n’est pas de reconstruire le schéma SQL initial,
mais de proposer un modèle relationnel adapté
au niveau de granularité et à l’usage métier considéré.


In [72]:
# MIGRATION INVERSE 1
# Depuis la dénormalisation 1 — Event détaillé
# Chaque document MongoDB représente un événement complet.
# On peut donc reconstruire une base SQL opérationnelle normalisée.
# Implémentation SQL + insertion

import sqlite3

conn = sqlite3.connect("sql_inverse_1_event.db")
cursor = conn.cursor()

cursor.executescript("""
CREATE TABLE IF NOT EXISTS city (
    city_id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT UNIQUE
);

CREATE TABLE IF NOT EXISTS user (
    user_id INTEGER PRIMARY KEY,
    name TEXT,
    age INTEGER,
    genre TEXT,
    subscription TEXT
);

CREATE TABLE IF NOT EXISTS vehicle (
    vehicle_id INTEGER PRIMARY KEY AUTOINCREMENT,
    type TEXT,
    energy TEXT
);

CREATE TABLE IF NOT EXISTS station (
    station_id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT,
    city_id INTEGER
);

CREATE TABLE IF NOT EXISTS environment (
    environment_id INTEGER PRIMARY KEY AUTOINCREMENT,
    temperature REAL,
    rain INTEGER,
    pm10 REAL,
    traffic_volume INTEGER
);

CREATE TABLE IF NOT EXISTS payment (
    payment_id INTEGER PRIMARY KEY AUTOINCREMENT,
    amount REAL,
    method TEXT
);

CREATE TABLE IF NOT EXISTS event (
    event_id INTEGER PRIMARY KEY,
    datetime TEXT,
    zone TEXT,
    risque_accident INTEGER,
    distance_km REAL,
    duration_min REAL,
    user_id INTEGER,
    vehicle_id INTEGER,
    station_start_id INTEGER,
    station_end_id INTEGER,
    environment_id INTEGER,
    payment_id INTEGER
);
""")

for doc in collection_event_detail.find():
    cursor.execute("INSERT OR IGNORE INTO city(name) VALUES (?)", (doc["ville"],))
    cursor.execute("SELECT city_id FROM city WHERE name=?", (doc["ville"],))
    city_id = cursor.fetchone()[0]

    u = doc["utilisateur"]
    cursor.execute(
        "INSERT OR IGNORE INTO user VALUES (?, ?, ?, ?, ?)",
        (u["id_utilisateur"], u["nom_utilisateur"], u["age"], u["genre"], u["abonnement"])
    )

    v = doc["vehicule"]
    cursor.execute("INSERT INTO vehicle(type, energy) VALUES (?, ?)", (v["type"], v["energie"]))
    vehicle_id = cursor.lastrowid

    for s in ["station_depart", "station_arrivee"]:
        cursor.execute(
            "INSERT INTO station(name, city_id) VALUES (?, ?)",
            (doc["trajet"][s], city_id)
        )

    env = doc["environnement"]
    cursor.execute(
        "INSERT INTO environment VALUES (NULL, ?, ?, ?, ?)",
        (env["temperature"], env["pluie"], env["pm10"], env["volume_trafic"])
    )
    env_id = cursor.lastrowid

    p = doc["paiement"]
    cursor.execute(
        "INSERT INTO payment VALUES (NULL, ?, ?)",
        (p["montant"], p["moyen"])
    )
    pay_id = cursor.lastrowid

    cursor.execute("""
        INSERT INTO event VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        doc["event_id"],
        doc["date_heure"],
        doc["zone"],
        doc["risque_accident"],
        doc["trajet"]["distance_km"],
        doc["trajet"]["duree_min"],
        u["id_utilisateur"],
        vehicle_id,
        cursor.lastrowid - 2,
        cursor.lastrowid - 1,
        env_id,
        pay_id
    ))

conn.commit()
conn.close()


In [73]:
sqlite3.connect("sql_inverse_1_event.db").cursor().execute(
    "SELECT COUNT(*) FROM event"
).fetchone()
# Validation du nombre d'événements insérés

(1500,)

La migration inverse 1 Event détaillé repose sur des documents MongoDB
contenant des événements complets et non agrégés.

Chaque document représentant un événement unique,
il est possible de reconstruire un modèle relationnel
opérationnel et normalisé.

Les entités métier (utilisateur, véhicule, environnement, paiement)
sont séparées dans des tables dédiées afin d’éviter les redondances
et de garantir la cohérence des données.

Cette migration inverse est la seule à permettre
une reconstruction complète et fidèle d’un modèle relationnel
événementiel.


In [74]:
# MIGRATION INVERSE 2
# Depuis la dénormalisation 2 — Sécurité par zone
# IMPLÉMENTATION SQL + INSERTION
conn = sqlite3.connect("sql_inverse_2_zone.db")
cursor = conn.cursor()

cursor.executescript("""
CREATE TABLE zone (
    zone_id INTEGER PRIMARY KEY AUTOINCREMENT,
    nom TEXT UNIQUE
);

CREATE TABLE zone_stats (
    zone_id INTEGER,
    risque_moyen REAL,
    pollution_moyenne REAL,
    temperature_moyenne REAL,
    frequence_pluie REAL,
    nombre_evenements INTEGER,
    alerte TEXT,
    date_calcul TEXT
);

CREATE TABLE zone_vehicle_stats (
    zone_id INTEGER,
    vehicle_type TEXT,
    nombre INTEGER,
    pourcentage REAL
);
""")

for doc in collection_zone.find():
    cursor.execute("INSERT OR IGNORE INTO zone(nom) VALUES (?)", (doc["zone"],))
    cursor.execute("SELECT zone_id FROM zone WHERE nom=?", (doc["zone"],))
    zid = cursor.fetchone()[0]

    s = doc["statistiques"]
    cursor.execute("""
        INSERT INTO zone_stats VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        zid,
        s["risque_accident_moyen"],
        s["pollution_pm10_moyenne"],
        s["temperature_moyenne"],
        s["frequence_pluie"],
        s["nombre_evenements_total"],
        doc["alerte_securite"],
        doc["derniere_mise_a_jour"]
    ))

conn.commit()
conn.close()


In [75]:
sqlite3.connect("sql_inverse_2_zone.db").cursor().execute(
    "SELECT * FROM zone_stats LIMIT 5"
).fetchall()
#Validation des données insérées

[(1, 2.59, 21.35, 2.14, 0.59, 298, 'MOYENNE', '2026-01-31T16:10:49.264752'),
 (2, 2.49, 21.14, 2.13, 0.61, 292, 'MOYENNE', '2026-01-31T16:10:49.264974'),
 (3, 2.47, 20.98, 2.19, 0.6, 314, 'MOYENNE', '2026-01-31T16:10:49.265188'),
 (4, 2.56, 21.0, 2.22, 0.62, 286, 'MOYENNE', '2026-01-31T16:10:49.265365'),
 (5, 2.45, 21.14, 2.14, 0.55, 310, 'MOYENNE', '2026-01-31T16:10:49.265529')]

La migration inverse 2 Sécurité par zone s’appuie sur des données agrégées
par zone géographique.

Les événements unitaires ayant disparu lors de la dénormalisation,
il n’est pas possible de reconstruire un modèle opérationnel.

La normalisation inverse produit donc un modèle relationnel
analytique, centré sur les indicateurs de sécurité par zone.

Ce schéma est volontairement orienté lecture et analyse,
et correspond aux usages de pilotage et de prise de décision.


In [76]:
# MIGRATION INVERSE 3
# Depuis la dénormalisation 3 — Marketing utilisateurs
# IMPLÉMENTATION SQL + INSERTION
conn = sqlite3.connect("sql_inverse_3_users.db")
cursor = conn.cursor()

cursor.executescript("""
CREATE TABLE user (
    user_id INTEGER PRIMARY KEY,
    nom TEXT,
    age INTEGER,
    genre TEXT,
    abonnement TEXT
);

CREATE TABLE user_stats (
    user_id INTEGER,
    nombre_trajets INTEGER,
    distance_moyenne REAL,
    depenses_totales REAL,
    score REAL
);
""")

for doc in collection_users.find():
    cursor.execute(
        "INSERT INTO user VALUES (?, ?, ?, ?, ?)",
        (doc["id_utilisateur"], doc["nom_utilisateur"], doc["age"], doc["genre"], doc["abonnement_actuel"])
    )

    s = doc["statistiques_usage"]
    cursor.execute(
        "INSERT INTO user_stats VALUES (?, ?, ?, ?, ?)",
        (doc["id_utilisateur"], s["nombre_trajets"], s["distance_moyenne_km"], s["depenses_totales_euros"], doc["score_engagement"])
    )

conn.commit()
conn.close()


In [77]:
sqlite3.connect("sql_inverse_3_users.db").cursor().execute(
    "SELECT * FROM user_stats LIMIT 5"
).fetchall()


[(192, 5, 8.05, 20.17, 1.4),
 (142, 6, 5.99, 13.14, 1.8),
 (48, 7, 6.68, 20.41, 2.2),
 (247, 2, 2.32, 6.58, 0.6),
 (149, 8, 5.43, 30.38, 2.4)]

La migration inverse 3 Marketing utilisateurs est construite à partir de profils
utilisateurs synthétiques, contenant des statistiques d’usage
et des préférences calculées.

Le modèle relationnel obtenu est orienté marketing et CRM :
les informations stables de l’utilisateur sont séparées
des données statistiques et comportementales.

Cette normalisation facilite la segmentation, l’analyse
des comportements et l’exploitation par des outils de
relation client.


In [78]:
# MIGRATION INVERSE 4
# Depuis la dénormalisation 4 — Analytics global
# IMPLÉMENTATION SQL + INSERTION
import sqlite3

conn = sqlite3.connect("sql_inverse_4_global.db")
cursor = conn.cursor()

cursor.executescript("""
CREATE TABLE IF NOT EXISTS periode (
    periode_id INTEGER PRIMARY KEY AUTOINCREMENT,
    periode TEXT,
    derniere_mise_a_jour TEXT
);

CREATE TABLE IF NOT EXISTS global_kpi (
    periode_id INTEGER,
    nombre_trajets INTEGER,
    distance_totale_km REAL,
    duree_moyenne_min REAL,
    chiffre_affaires_total REAL,
    risque_moyen REAL,
    pollution_moyenne_pm10 REAL,
    taux_pluie REAL,
    volume_trafic_moyen REAL
);

CREATE TABLE IF NOT EXISTS global_vehicule_stats (
    periode_id INTEGER,
    type_vehicule TEXT,
    nombre_trajets INTEGER,
    pourcentage REAL
);
""")
conn.commit()


In [79]:

for doc in collection_analytics.find():

    # 1️⃣ PERIODE
    cursor.execute("""
        INSERT INTO periode (periode, derniere_mise_a_jour)
        VALUES (?, ?)
    """, (
        doc["periode"],
        doc["derniere_mise_a_jour"]
    ))
    periode_id = cursor.lastrowid

    # 2️⃣ KPI GLOBAUX
    k = doc["kpi_globaux"]

    cursor.execute("""
        INSERT INTO global_kpi VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        periode_id,
        k["nombre_trajets"],
        k["distance_totale_km"],
        k["duree_moyenne_min"],
        k["chiffre_affaires_total"],
        k["risque_moyen"],
        k["pollution_moyenne_pm10"],
        doc["indicateurs_securite"]["taux_pluie"],
        doc["impact_environnemental"]["volume_trafic_moyen"]
    ))

    # 3️⃣ RÉPARTITION DES VÉHICULES
    for v in doc["repartition_vehicules"]:
        cursor.execute("""
            INSERT INTO global_vehicule_stats VALUES (?, ?, ?, ?)
        """, (
            periode_id,
            v["type"],
            v["nombre_trajets"],
            v["pourcentage"]
        ))

conn.commit()
conn.close()


In [80]:
conn = sqlite3.connect("sql_inverse_4_global.db")
cursor = conn.cursor()

cursor.execute("""
SELECT p.periode, g.nombre_trajets, g.chiffre_affaires_total
FROM global_kpi g
JOIN periode p ON g.periode_id = p.periode_id
""").fetchall()
# Validation des données insérées

[('2025-10', 1500, 5370.23)]

La migration inverse 4 Analytics global produit un modèle relationnel décisionnel
adapté au pilotage global.

Les données étant déjà agrégées par période,
il n’est ni possible ni pertinent de reconstruire
un modèle événementiel.

Le schéma obtenu est volontairement simple,
lisible et directement exploitable pour l’analyse stratégique.


##### SYNTHÈSE GLOBALE
Plus les données sont dénormalisées et agrégées,
plus la normalisation inverse s’éloigne d’un modèle
opérationnel pour se rapprocher de modèles analytiques
ou décisionnels.

Chaque migration inverse est donc adaptée
au niveau d’information réellement disponible.


### ORM (Object Relational Mapping)

Cette partie vise à reproduire l’ensemble du travail réalisé précédemment
en utilisant un ORM.

L’objectif n’est pas de modifier les modèles ou les usages,
mais de démontrer que la logique métier et les migrations
peuvent être conservées indépendamment du mode d’accès aux données.

L’ORM permet d’abstraire les requêtes SQL
au profit de manipulations d’objets Python.

Toutes les étapes réalisées précédemment
(normalisation, migrations, dénormalisations et migrations inverses)
sont reproduites à l’identique,
afin de comparer une implémentation SQL/DAO
avec une implémentation orientée objet.


Le fichier CSV constitue la source unique de données
pour l’ensemble du projet.

Il est utilisé comme point d’entrée
aussi bien pour la version SQL/DAO
que pour la version ORM.

Cela garantit une comparaison équitable
entre les différentes approches techniques.


In [81]:
import pandas as pd

dfo = pd.read_csv("donnees_smartcity.csv")
dfo.head()

,id_evenement,date_heure,ville,zone,id_utilisateur,id_appareil,type_vehicule,energie,station_depart,station_arrivee,...,pluie,pm10,volume_trafic,montant_paiement,moyen_paiement,nom_utilisateur,age,abonnement,genre,risque_accident
0,1,2025-10-01 00:00,Paris,Sud,192,APP-3143,Velo,Electrique,4,56,...,0,16,148,3.93,Mobile,Lucas,29,Premium,homme,2
1,2,2025-10-01 00:20,Paris,Sud,142,APP-1823,Trottinette,Humaine,33,43,...,1,22,174,1.78,Mobile,Nina,29,Premium,femme,3
2,3,2025-10-01 00:40,Paris,Nord,48,APP-3836,Velo,Humaine,38,35,...,1,19,188,5.31,Carte,Lucas,41,Premium,homme,1
3,4,2025-10-01 01:00,Paris,Est,247,APP-9612,Bus,Electrique,48,34,...,1,21,141,1.35,Carte,Hugo,26,Standard,femme,4
4,5,2025-10-01 01:20,Paris,Est,149,APP-4076,Bus,Humaine,60,24,...,0,16,137,3.05,Mobile,Emma,63,Standard,homme,2


##### NORMALISATION
La normalisation relationnelle utilisée dans cette partie
est strictement identique à celle définie précédemment.

Elle repose sur l’identification des entités métier
et sur l’élimination des redondances
présentes dans les données brutes.


Chaque table issue de la normalisation SQL
est représentée par une classe ORM.

Les relations entre tables sont traduites
par des relations entre objets Python.


##### Entités métier identifiées

City : une ville existe indépendamment des événements

Station : une station appartient à une ville

User : un utilisateur peut avoir plusieurs événements

Vehicle : un véhicule peut être partagé

Device : identifiant technique de l’appareil

Environment : conditions ponctuelles

Payment : paiement associé à un événement

Event : entité centrale

##### IMPLÉMENTATION ORM (Les tables)

In [82]:
from sqlalchemy import (
    create_engine, Column, Integer, String,
    Float, ForeignKey
)
from sqlalchemy.orm import declarative_base, relationship

Base = declarative_base()


In [83]:
# City
class City(Base):
    __tablename__ = "city"
    city_id = Column(Integer, primary_key=True)
    name = Column(String, unique=True)

    stations = relationship("Station", back_populates="city")


In [84]:
# Station
class Station(Base):
    __tablename__ = "station"
    station_id = Column(Integer, primary_key=True)
    name = Column(String)
    city_id = Column(Integer, ForeignKey("city.city_id"))

    city = relationship("City", back_populates="stations")


In [85]:
# Device
class Device(Base):
    __tablename__ = "device"
    device_id = Column(String, primary_key=True)


In [86]:
# User
class User(Base):
    __tablename__ = "user"
    user_id = Column(Integer, primary_key=True)
    name = Column(String)
    age = Column(Integer)
    genre = Column(String)
    subscription = Column(String)

    events = relationship("Event", back_populates="user")


In [87]:
# Vehicule
class Vehicle(Base):
    __tablename__ = "vehicle"
    vehicle_id = Column(Integer, primary_key=True)
    type = Column(String)
    energy = Column(String)

    events = relationship("Event", back_populates="vehicle")


In [88]:
# Environnement
class Environment(Base):
    __tablename__ = "environment"
    environment_id = Column(Integer, primary_key=True)
    temperature = Column(Float)
    rain = Column(Integer)
    pm10 = Column(Float)
    traffic_volume = Column(Integer)


In [89]:
# Payment
class Payment(Base):
    __tablename__ = "payment"
    payment_id = Column(Integer, primary_key=True)
    amount = Column(Float)
    method = Column(String)


In [90]:
# Event (Centrale)
class Event(Base):
    __tablename__ = "event"

    event_id = Column(Integer, primary_key=True)
    datetime = Column(String)
    zone = Column(String)
    risque_accident = Column(Integer)
    distance_km = Column(Float)
    duration_min = Column(Float)

    user_id = Column(Integer, ForeignKey("user.user_id"))
    vehicle_id = Column(Integer, ForeignKey("vehicle.vehicle_id"))
    environment_id = Column(Integer, ForeignKey("environment.environment_id"))
    payment_id = Column(Integer, ForeignKey("payment.payment_id"))
    device_id = Column(String, ForeignKey("device.device_id"))
    station_start_id = Column(Integer, ForeignKey("station.station_id"))
    station_end_id = Column(Integer, ForeignKey("station.station_id"))

    user = relationship("User", back_populates="events")
    vehicle = relationship("Vehicle", back_populates="events")
    environment = relationship("Environment")
    payment = relationship("Payment")
    device = relationship("Device")
    station_start = relationship("Station", foreign_keys=[station_start_id])
    station_end = relationship("Station", foreign_keys=[station_end_id])


##### CRÉATION DE LA BASE ORM

In [91]:
from sqlalchemy.orm import sessionmaker

engine = create_engine("sqlite:///orm_smartcity.db")
Base.metadata.create_all(engine)

Session = sessionmaker(bind=engine)
session = Session()


##### INSERTION ORM (CSV → BASE NORMALISÉE)

Cette étape consiste à insérer les données issues du fichier CSV
dans la base relationnelle normalisée,
en utilisant exclusivement l’ORM.

L’objectif est de rejouer la normalisation SQL définie précédemment,
tout en garantissant la cohérence des données
et l’intégrité des relations entre les entités.


In [92]:
session.query(Event).delete()
session.query(Payment).delete()
session.query(Environment).delete()
session.query(Station).delete()
session.query(Vehicle).delete()
session.query(User).delete()
session.query(Device).delete()
session.query(City).delete()
session.commit()


In [93]:
cities = {}
stations = {}
users = {}
vehicles = {}
devices = {}

for _, row in df.iterrows():

    # ───────────────── CITY ─────────────────
    city_name = row["ville"]
    if city_name not in cities:
        city = City(name=city_name)
        session.add(city)
        session.flush()
        cities[city_name] = city
    city = cities[city_name]

    # ───────────────── DEVICE ───────────────
    device_id = str(row["id_appareil"])
    if device_id not in devices:
        device = Device(device_id=device_id)
        session.add(device)
        devices[device_id] = device
    device = devices[device_id]

    # ───────────────── USER ─────────────────
    user_id = row["id_utilisateur"]
    if user_id not in users:
        user = User(
            user_id=user_id,
            name=row["nom_utilisateur"],
            age=row["age"],
            genre=row["genre"],
            subscription=row["abonnement"]
        )
        session.add(user)
        users[user_id] = user
    user = users[user_id]

    # ───────────────── VEHICLE ──────────────
    v_key = (row["type_vehicule"], row["energie"])
    if v_key not in vehicles:
        vehicle = Vehicle(
            type=row["type_vehicule"],
            energy=row["energie"]
        )
        session.add(vehicle)
        session.flush()
        vehicles[v_key] = vehicle
    vehicle = vehicles[v_key]

    # ───────────────── STATIONS ─────────────
    for col in ["station_depart", "station_arrivee"]:
        s_key = (row[col], city.city_id)
        if s_key not in stations:
            station = Station(name=row[col], city=city)
            session.add(station)
            session.flush()
            stations[s_key] = station

    station_start = stations[(row["station_depart"], city.city_id)]
    station_end = stations[(row["station_arrivee"], city.city_id)]

    # ───────────────── ENVIRONMENT ──────────
    environment = Environment(
        temperature=row["temperature"],
        rain=row["pluie"],
        pm10=row["pm10"],
        traffic_volume=row["volume_trafic"]
    )
    session.add(environment)
    session.flush()

    # ───────────────── PAYMENT ──────────────
    payment = Payment(
        amount=row["montant_paiement"],
        method=row["moyen_paiement"]
    )
    session.add(payment)
    session.flush()

    # ───────────────── EVENT ────────────────
    event = Event(
        event_id=row["id_evenement"],
        datetime=row["date_heure"],
        zone=row["zone"],
        risque_accident=row["risque_accident"],
        distance_km=row["distance_km"],
        duration_min=row["duree_min"],
        user=user,
        vehicle=vehicle,
        environment=environment,
        payment=payment,
        device=device,
        station_start=station_start,
        station_end=station_end
    )
    session.add(event)

session.commit()


L’insertion ORM reproduit strictement la normalisation SQL.
La différence réside uniquement dans l’utilisation d’objets
au lieu de requêtes SQL explicites.


Avant de poursuivre les migrations et dénormalisations,
il est indispensable de valider que la base relationnelle
créée via l’ORM est correcte, complète et cohérente.

Cette étape permet de s’assurer que l’ORM a bien rejoué
la normalisation SQL définie précédemment.

In [104]:
import sqlite3

conn = sqlite3.connect("orm_smartcity.db")
cursor = conn.cursor()

tables = [
    "city", "station", "device", "user",
    "vehicle", "environment", "payment", "event"
]

for t in tables:
    print(t, cursor.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0])


city 1
station 60
device 1379
user 250
vehicle 12
environment 1500
payment 1500
event 1500


Toutes les tables issues du DEA sont présentes.
L’ORM a correctement généré le schéma relationnel attendu.


In [106]:
cursor.execute("""
SELECT COUNT(DISTINCT c.city_id), COUNT(e.event_id)
FROM city c
JOIN station s ON s.city_id = c.city_id
JOIN event e ON e.station_start_id = s.station_id
""").fetchall()


[(1, 1500)]

Nos données concerne le smartcity de la ville de Paris donc c'est normal qu'on est un seule ville avec plusieurs evenement ce qui justifie le fait que Le fichier CSV ne contient qu’une seule ville (Paris).

La normalisation relationnelle consiste à stocker
chaque information unique une seule fois,
puis à la référencer via des clés étrangères.

Il est donc normal que la table city ne contienne
qu’une seule ligne, même si la ville apparaît
dans l’ensemble des événements.


In [96]:
cursor.execute("PRAGMA table_info(event)").fetchall()
# Structure de la table event

[(0, 'event_id', 'INTEGER', 1, None, 1),
 (1, 'datetime', 'VARCHAR', 0, None, 0),
 (2, 'zone', 'VARCHAR', 0, None, 0),
 (3, 'risque_accident', 'INTEGER', 0, None, 0),
 (4, 'distance_km', 'FLOAT', 0, None, 0),
 (5, 'duration_min', 'FLOAT', 0, None, 0),
 (6, 'user_id', 'INTEGER', 0, None, 0),
 (7, 'vehicle_id', 'INTEGER', 0, None, 0),
 (8, 'environment_id', 'INTEGER', 0, None, 0),
 (9, 'payment_id', 'INTEGER', 0, None, 0),
 (10, 'device_id', 'VARCHAR', 0, None, 0),
 (11, 'station_start_id', 'INTEGER', 0, None, 0),
 (12, 'station_end_id', 'INTEGER', 0, None, 0)]

In [97]:
cursor.execute("PRAGMA table_info(user)").fetchall()
# Structure de la table user

[(0, 'user_id', 'INTEGER', 1, None, 1),
 (1, 'name', 'VARCHAR', 0, None, 0),
 (2, 'age', 'INTEGER', 0, None, 0),
 (3, 'genre', 'VARCHAR', 0, None, 0),
 (4, 'subscription', 'VARCHAR', 0, None, 0)]

In [98]:
cursor.execute("PRAGMA table_info(city)").fetchall()
# Structure de la table city

[(0, 'city_id', 'INTEGER', 1, None, 1), (1, 'name', 'VARCHAR', 0, None, 0)]

In [99]:
cursor.execute("PRAGMA table_info(device)").fetchall()
# Structure de la table device

[(0, 'device_id', 'VARCHAR', 1, None, 1)]

In [100]:
cursor.execute("PRAGMA table_info(station)").fetchall()
# Structure de la table station

[(0, 'station_id', 'INTEGER', 1, None, 1),
 (1, 'name', 'VARCHAR', 0, None, 0),
 (2, 'city_id', 'INTEGER', 0, None, 0)]

In [102]:
cursor.execute("PRAGMA table_info(payment)").fetchall()
# Structure de la table payment

[(0, 'payment_id', 'INTEGER', 1, None, 1),
 (1, 'amount', 'FLOAT', 0, None, 0),
 (2, 'method', 'VARCHAR', 0, None, 0)]

##### LECTURE DES DONNÉES AVEC ORM

Cette étape vise à démontrer comment l’ORM permet
de lire et de manipuler les données relationnelles
sans écrire de requêtes SQL explicites.

Elle remplace le rôle des DAO SQL,
tout en conservant la même logique métier.


In [ ]:
events = session.query(Event).limit(5).all()
events
# La requête retourne une liste d’objets Event.
# Chaque objet correspond à une ligne de la table event, avec ses attributs et ses relations associées.



In [108]:
e = session.query(Event).first()

print("Date :", e.datetime)
print("Utilisateur :", e.user.name)
print("Véhicule :", e.vehicle.type, e.vehicle.energy)
print("Ville :", e.station_start.city.name)
print("Paiement :", e.payment.amount, e.payment.method)


Date : 2025-10-01 00:00
Utilisateur : Lucas
Véhicule : Velo Electrique
Ville : Paris
Paiement : 3.93 Mobile


Grâce aux relations ORM, il est possible
d’accéder directement aux entités liées
sans écrire de jointures SQL.

Les relations sont gérées automatiquement
par l’ORM.


In [ ]:
nb_events = session.query(Event).count()
nb_events
# nombre d’événements

1500

La lecture ORM remplace les DAO SQL
en proposant une navigation orientée objet.

Les jointures SQL sont remplacées
par des relations explicites entre classes.

Cette approche améliore la lisibilité du code
et réduit les risques d’erreurs liées aux jointures.


#### MIGRATION ORM → NoSQL

Cette étape consiste à produire des dénormalisations NoSQL
comme celles réalisées précédemment,
mais en utilisant l’ORM comme source relationnelle.

L’objectif est de démontrer que les dénormalisations
reposent sur la structure des données
et non sur la technologie d’accès au SQL.


##### DÉNORMALISATION N°1: USAGE (métier)
Je suis un analyste data opérationnel.
Je souhaite accéder rapidement à tous les détails
d’un événement sans effectuer de jointures SQL complexes.


Source relationnelle :
- Table centrale : Event
- Tables liées : User, Vehicle, Station, City, Environment, Payment, Device

Accès via ORM :
session.query(Event) + navigation par relations


# SCHÉMA DU DOCUMENT NoSQL
```
{
  "event_id": 1024,
  "date_heure": "2025-10-12 08:45",
  "zone": "Centre",
  "risque_accident": 3,

  "utilisateur": {
    "id_utilisateur": 192,
    "nom": "Lucas",
    "age": 29,
    "genre": "homme",
    "abonnement": "Premium"
  },

  "vehicule": {
    "type": "Velo",
    "energie": "Electrique"
  },

  "trajet": {
    "station_depart": "Gare Nord",
    "station_arrivee": "République",
    "ville": "Paris",
    "distance_km": 5.2,
    "duree_min": 14.3
  },

  "environnement": {
    "temperature": 12.5,
    "pluie": 1,
    "pm10": 18.4,
    "volume_trafic": 220
  },

  "paiement": {
    "montant": 3.5,
    "moyen": "Mobile"
  },

  "device_id": "A45F9"
}


Cette dénormalisation regroupe toutes les informations
liées à un événement dans un seul document.

Elle est adaptée à des usages de consultation fréquente
où la priorité est la rapidité de lecture.


##### Implémentation ORM → MongoDB

In [111]:
from pymongo import MongoClient

client = MongoClient()
db_mongo = client["smartcity_orm"]

collection_event = db_mongo["event_detail"]
collection_event.delete_many({})

events = session.query(Event).all()

documents = []

for e in events:
    documents.append({
        "event_id": e.event_id,
        "date_heure": e.datetime,
        "zone": e.zone,
        "risque_accident": e.risque_accident,

        "utilisateur": {
            "id_utilisateur": e.user.user_id,
            "nom": e.user.name,
            "age": e.user.age,
            "genre": e.user.genre,
            "abonnement": e.user.subscription
        },

        "vehicule": {
            "type": e.vehicle.type,
            "energie": e.vehicle.energy
        },

        "trajet": {
            "station_depart": e.station_start.name,
            "station_arrivee": e.station_end.name,
            "ville": e.station_start.city.name,
            "distance_km": e.distance_km,
            "duree_min": e.duration_min
        },

        "environnement": {
            "temperature": e.environment.temperature,
            "pluie": e.environment.rain,
            "pm10": e.environment.pm10,
            "volume_trafic": e.environment.traffic_volume
        },

        "paiement": {
            "montant": e.payment.amount,
            "moyen": e.payment.method
        },

        "device_id": e.device.device_id
    })

collection_event.insert_many(documents)


InsertManyResult([ObjectId('697e323361deb6479274f781'), ObjectId('697e323361deb6479274f782'), ObjectId('697e323361deb6479274f783'), ObjectId('697e323361deb6479274f784'), ObjectId('697e323361deb6479274f785'), ObjectId('697e323361deb6479274f786'), ObjectId('697e323361deb6479274f787'), ObjectId('697e323361deb6479274f788'), ObjectId('697e323361deb6479274f789'), ObjectId('697e323361deb6479274f78a'), ObjectId('697e323361deb6479274f78b'), ObjectId('697e323361deb6479274f78c'), ObjectId('697e323361deb6479274f78d'), ObjectId('697e323361deb6479274f78e'), ObjectId('697e323361deb6479274f78f'), ObjectId('697e323361deb6479274f790'), ObjectId('697e323361deb6479274f791'), ObjectId('697e323361deb6479274f792'), ObjectId('697e323361deb6479274f793'), ObjectId('697e323361deb6479274f794'), ObjectId('697e323361deb6479274f795'), ObjectId('697e323361deb6479274f796'), ObjectId('697e323361deb6479274f797'), ObjectId('697e323361deb6479274f798'), ObjectId('697e323361deb6479274f799'), ObjectId('697e323361deb6479274f7

In [118]:
from pprint import pprint

for doc in collection_event.find().limit(2):
    pprint(doc)


{'_id': ObjectId('697e323361deb6479274f781'),
 'date_heure': '2025-10-01 00:00',
 'device_id': 'APP-3143',
 'environnement': {'pluie': 0,
                   'pm10': 16.0,
                   'temperature': 2.0,
                   'volume_trafic': 148},
 'event_id': 1,
 'paiement': {'montant': 3.93, 'moyen': 'Mobile'},
 'risque_accident': 2,
 'trajet': {'distance_km': 4.66,
            'duree_min': 3.0,
            'station_arrivee': '56',
            'station_depart': '4',
            'ville': 'Paris'},
 'utilisateur': {'abonnement': 'Premium',
                 'age': 29,
                 'genre': 'homme',
                 'id_utilisateur': 192,
                 'nom': 'Lucas'},
 'vehicule': {'energie': 'Electrique', 'type': 'Velo'},
 'zone': 'Sud'}
{'_id': ObjectId('697e323361deb6479274f782'),
 'date_heure': '2025-10-01 00:20',
 'device_id': 'APP-1823',
 'environnement': {'pluie': 1,
                   'pm10': 22.0,
                   'temperature': 2.3,
                   'volume_traf

Chaque document représente un événement unique.

Toutes les informations nécessaires sont présentes :
- données utilisateur
- véhicule
- trajet
- environnement
- paiement

Les données sont regroupées de manière cohérente
et directement exploitables sans jointures.


Cette dénormalisation permet un accès immédiat
à toutes les informations d’un événement.

Elle est adaptée à des usages de consultation
et d’analyse opérationnelle.


##### DÉNORMALISATION N°2 — SÉCURITÉ PAR ZONE

##### Usage métier
Je suis responsable de la sécurité urbaine.
Je consulte régulièrement les indicateurs de risque
par zone afin d’adapter les actions de prévention.


##### SOURCE
- Event (zone, risque_accident)
- Environment (pluie, pm10, température)
- Agrégations calculées côté application


##### SCHÉMA DU DOCUMENT
```
{
  "zone": "Nord",
  "statistiques": {
    "nombre_evenements_total": 287,
    "risque_accident_moyen": 2.45,
    "pollution_pm10_moyenne": 19.8,
    "temperature_moyenne": 11.2,
    "frequence_pluie": 0.73
  },
  "alerte_securite": "MOYENNE",
  "derniere_mise_a_jour": "2026-01-30T10:30:00"
}


In [114]:
# Implémentation ORM → MongoDB
from collections import defaultdict
from statistics import mean
from datetime import datetime

collection_zone = db_mongo["zone_security"]
collection_zone.delete_many({})

zones = defaultdict(list)

for e in events:
    zones[e.zone].append(e)

documents = []

for zone, evts in zones.items():
    documents.append({
        "zone": zone,
        "statistiques": {
            "nombre_evenements_total": len(evts),
            "risque_accident_moyen": round(mean(e.risque_accident for e in evts), 2),
            "pollution_pm10_moyenne": round(mean(e.environment.pm10 for e in evts), 2),
            "temperature_moyenne": round(mean(e.environment.temperature for e in evts), 2),
            "frequence_pluie": round(mean(e.environment.rain for e in evts), 2)
        },
        "alerte_securite": (
            "ELEVEE" if mean(e.risque_accident for e in evts) > 3
            else "MOYENNE" if mean(e.risque_accident for e in evts) > 2
            else "FAIBLE"
        ),
        "derniere_mise_a_jour": datetime.utcnow().isoformat()
    })

collection_zone.insert_many(documents)


C:\Users\Pc\AppData\Local\Temp\ipykernel_24908\885077629.py:31: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "derniere_mise_a_jour": datetime.utcnow().isoformat()


InsertManyResult([ObjectId('697e4ae461deb6479274fd5d'), ObjectId('697e4ae461deb6479274fd5e'), ObjectId('697e4ae461deb6479274fd5f'), ObjectId('697e4ae461deb6479274fd60'), ObjectId('697e4ae461deb6479274fd61')], acknowledged=True)

In [119]:
for doc in collection_zone.find():
    pprint(doc)


{'_id': ObjectId('697e4ae461deb6479274fd5d'),
 'alerte_securite': 'MOYENNE',
 'derniere_mise_a_jour': '2026-01-31T18:33:08.648696',
 'statistiques': {'frequence_pluie': 0.59,
                  'nombre_evenements_total': 298,
                  'pollution_pm10_moyenne': 21.35,
                  'risque_accident_moyen': 2.59,
                  'temperature_moyenne': 2.14},
 'zone': 'Sud'}
{'_id': ObjectId('697e4ae461deb6479274fd5e'),
 'alerte_securite': 'MOYENNE',
 'derniere_mise_a_jour': '2026-01-31T18:33:08.652810',
 'statistiques': {'frequence_pluie': 0.61,
                  'nombre_evenements_total': 292,
                  'pollution_pm10_moyenne': 21.14,
                  'risque_accident_moyen': 2.49,
                  'temperature_moyenne': 2.13},
 'zone': 'Nord'}
{'_id': ObjectId('697e4ae461deb6479274fd5f'),
 'alerte_securite': 'MOYENNE',
 'derniere_mise_a_jour': '2026-01-31T18:33:08.656494',
 'statistiques': {'frequence_pluie': 0.6,
                  'nombre_evenements_total': 31

Chaque document correspond à une zone géographique.

Les statistiques sont agrégées correctement :
- nombre total d’événements
- risque moyen
- pollution moyenne
- conditions météo

Le champ alerte_securite synthétise l’information
de manière immédiatement exploitable.


Les événements sont agrégés par zone
pour produire des indicateurs synthétique

##### DÉNORMALISATION N°3 — Profil utilisateur marketing

##### USAGE
```
Je suis responsable marketing.
Je veux analyser rapidement le comportement
et les préférences de chaque utilisateur.


##### SOURCE
- User (profil)
- Event (distance, durée)
- Vehicle (type)
- Payment (méthode)

##### SCHÉMA DU DOCUMENT
```
{
  "id_utilisateur": 192,
  "nom": "Lucas",
  "age": 29,
  "genre": "homme",
  "abonnement": "Premium",

  "statistiques_usage": {
    "nombre_trajets": 47,
    "distance_moyenne_km": 5.8,
    "duree_moyenne_min": 14.2
  },

  "preferences": {
    "vehicule_prefere": "Velo",
    "moyen_paiement_prefere": "Mobile"
  }
}


In [116]:
from collections import Counter

collection_user = db_mongo["user_marketing"]
collection_user.delete_many({})

users = defaultdict(list)

for e in events:
    users[e.user.user_id].append(e)

documents = []

for uid, evts in users.items():
    vehicules = [e.vehicle.type for e in evts]
    paiements = [e.payment.method for e in evts]

    documents.append({
        "id_utilisateur": uid,
        "nom": evts[0].user.name,
        "age": evts[0].user.age,
        "genre": evts[0].user.genre,
        "abonnement": evts[0].user.subscription,

        "statistiques_usage": {
            "nombre_trajets": len(evts),
            "distance_moyenne_km": round(mean(e.distance_km for e in evts), 2),
            "duree_moyenne_min": round(mean(e.duration_min for e in evts), 2)
        },

        "preferences": {
            "vehicule_prefere": Counter(vehicules).most_common(1)[0][0],
            "moyen_paiement_prefere": Counter(paiements).most_common(1)[0][0]
        }
    })

collection_user.insert_many(documents)


InsertManyResult([ObjectId('697e4c2761deb6479274fd62'), ObjectId('697e4c2761deb6479274fd63'), ObjectId('697e4c2761deb6479274fd64'), ObjectId('697e4c2761deb6479274fd65'), ObjectId('697e4c2761deb6479274fd66'), ObjectId('697e4c2761deb6479274fd67'), ObjectId('697e4c2761deb6479274fd68'), ObjectId('697e4c2761deb6479274fd69'), ObjectId('697e4c2761deb6479274fd6a'), ObjectId('697e4c2761deb6479274fd6b'), ObjectId('697e4c2761deb6479274fd6c'), ObjectId('697e4c2761deb6479274fd6d'), ObjectId('697e4c2761deb6479274fd6e'), ObjectId('697e4c2761deb6479274fd6f'), ObjectId('697e4c2761deb6479274fd70'), ObjectId('697e4c2761deb6479274fd71'), ObjectId('697e4c2761deb6479274fd72'), ObjectId('697e4c2761deb6479274fd73'), ObjectId('697e4c2761deb6479274fd74'), ObjectId('697e4c2761deb6479274fd75'), ObjectId('697e4c2761deb6479274fd76'), ObjectId('697e4c2761deb6479274fd77'), ObjectId('697e4c2761deb6479274fd78'), ObjectId('697e4c2761deb6479274fd79'), ObjectId('697e4c2761deb6479274fd7a'), ObjectId('697e4c2761deb6479274fd

In [120]:
for doc in collection_user.find().limit(3):
    pprint(doc)


{'_id': ObjectId('697e4c2761deb6479274fd62'),
 'abonnement': 'Premium',
 'age': 29,
 'genre': 'homme',
 'id_utilisateur': 192,
 'nom': 'Lucas',
 'preferences': {'moyen_paiement_prefere': 'Mobile',
                 'vehicule_prefere': 'Velo'},
 'statistiques_usage': {'distance_moyenne_km': 8.05,
                        'duree_moyenne_min': 13.6,
                        'nombre_trajets': 5}}
{'_id': ObjectId('697e4c2761deb6479274fd63'),
 'abonnement': 'Premium',
 'age': 29,
 'genre': 'femme',
 'id_utilisateur': 142,
 'nom': 'Nina',
 'preferences': {'moyen_paiement_prefere': 'Mobile', 'vehicule_prefere': 'Bus'},
 'statistiques_usage': {'distance_moyenne_km': 5.99,
                        'duree_moyenne_min': 14.0,
                        'nombre_trajets': 6}}
{'_id': ObjectId('697e4c2761deb6479274fd64'),
 'abonnement': 'Premium',
 'age': 41,
 'genre': 'homme',
 'id_utilisateur': 48,
 'nom': 'Lucas',
 'preferences': {'moyen_paiement_prefere': 'Carte',
                 'vehicule_prefere': '

Chaque document représente un utilisateur unique.

Les statistiques d’usage et les préférences
sont correctement calculées à partir des événements.

Ce format est parfaitement adapté
aux usages CRM et marketing.


Le profil utilisateur est stocké dans un document unique.

Cela facilite la segmentation marketing
et l’analyse comportementale,
sans nécessiter de jointures complexes.


##### DÉNORMALISATION N°4 — Analytics global

##### USAGE
```
Je suis décideur ou responsable stratégique.
Je souhaite une vue globale et synthétique
de l’activité du service.

#### SOURCE
- Ensemble des événements
- Agrégations globales calculées via l’ORM

##### SCHÉMA DU DOCUMENT
```
{
  "periode": "globale",
  "kpi_globaux": {
    "nombre_trajets": 1500,
    "distance_totale_km": 9314.02,
    "duree_moyenne_min": 13.47,
    "chiffre_affaires_total": 5370.23,
    "risque_moyen": 2.51,
    "pollution_moyenne_pm10": 21.12
  },
  "derniere_mise_a_jour": "2026-01-30T21:34:56"
}


In [117]:
collection_global = db_mongo["global_analytics"]
collection_global.delete_many({})

documents = [{
    "periode": "globale",
    "kpi_globaux": {
        "nombre_trajets": len(events),
        "distance_totale_km": round(sum(e.distance_km for e in events), 2),
        "duree_moyenne_min": round(mean(e.duration_min for e in events), 2),
        "chiffre_affaires_total": round(sum(e.payment.amount for e in events), 2),
        "risque_moyen": round(mean(e.risque_accident for e in events), 2),
        "pollution_moyenne_pm10": round(mean(e.environment.pm10 for e in events), 2)
    },
    "derniere_mise_a_jour": datetime.utcnow().isoformat()
}]

collection_global.insert_many(documents)


C:\Users\Pc\AppData\Local\Temp\ipykernel_24908\1788553507.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "derniere_mise_a_jour": datetime.utcnow().isoformat()


InsertManyResult([ObjectId('697e4d1161deb6479274fe5c')], acknowledged=True)

In [121]:
pprint(collection_global.find_one())


{'_id': ObjectId('697e4d1161deb6479274fe5c'),
 'derniere_mise_a_jour': '2026-01-31T18:42:25.460539',
 'kpi_globaux': {'chiffre_affaires_total': 5370.23,
                 'distance_totale_km': 9314.02,
                 'duree_moyenne_min': 13.47,
                 'nombre_trajets': 1500,
                 'pollution_moyenne_pm10': 21.12,
                 'risque_moyen': 2.51},
 'periode': 'globale'}


Ce document fournit une vue synthétique
de l’ensemble de l’activité.

Les KPI globaux sont cohérents
avec le volume total des événements
et les montants agrégés.


Cette dénormalisation fournit une vue macro
orientée pilotage stratégique.

Elle sacrifie la granularité
au profit de la lisibilité et de la synthèse.


Les quatre dénormalisations NoSQL ont été reproduites en utilisant l’ORM comme source relationnelle.

Cela démontre que les choix de dénormalisation
sont indépendants de la technologie SQL utilisée.

Chaque dénormalisation répond à un usage métier précis.

Le schéma du document est conçu
en fonction des besoins de lecture,
et non de la logique relationnelle.

La dénormalisation NoSQL est donc un choix fonctionnel de notre part
et non une simple transformation technique.


#### MIGRATIONS INVERSES NoSQL → ORM

Ici, on vise à retrouver depuis des modèles NoSQL dénormalisés les tables (ORM) possibles.
L’objectif n’est pas de reconstruire strictement
le schéma relationnel initial,
mais de proposer des normalisations cohérentes,
justifiées et adaptées aux usages métiers.


##### MIGRATION INVERSE N°1 Depuis la dénormalisation Événement détaillé 


##### Schéma relationnel cible
```
event_flat
---------
event_id (PK)
date_heure
zone
risque_accident
distance_km
duree_min
ville
station_depart
station_arrivee
user_id
user_nom
user_age
user_genre
abonnement
vehicule_type
vehicule_energie
temperature
pluie
pm10
volume_trafic
montant_paiement
moyen_paiement
device_id


Cette normalisation inverse privilégie
la simplicité de lecture et d’analyse.

Elle correspond à un schéma de type data mart,
adapté aux usages analytiques et exploratoires.


In [122]:
class EventFlat(Base):
    __tablename__ = "event_flat"

    event_id = Column(Integer, primary_key=True)
    date_heure = Column(String)
    zone = Column(String)
    risque_accident = Column(Integer)
    distance_km = Column(Float)
    duree_min = Column(Float)

    ville = Column(String)
    station_depart = Column(String)
    station_arrivee = Column(String)

    user_id = Column(Integer)
    user_nom = Column(String)
    user_age = Column(Integer)
    user_genre = Column(String)
    abonnement = Column(String)

    vehicule_type = Column(String)
    vehicule_energie = Column(String)

    temperature = Column(Float)
    pluie = Column(Integer)
    pm10 = Column(Float)
    volume_trafic = Column(Integer)

    montant_paiement = Column(Float)
    moyen_paiement = Column(String)

    device_id = Column(String)


In [124]:
# Création physique des tables ORM inverses
Base.metadata.create_all(engine)


##### Insertion ORM depuis MongoDB

In [125]:
collection = db_mongo["event_detail"]

session.query(EventFlat).delete()
session.commit()

for doc in collection.find():
    row = EventFlat(
        event_id=doc["event_id"],
        date_heure=doc["date_heure"],
        zone=doc["zone"],
        risque_accident=doc["risque_accident"],
        distance_km=doc["trajet"]["distance_km"],
        duree_min=doc["trajet"]["duree_min"],
        ville=doc["trajet"]["ville"],
        station_depart=doc["trajet"]["station_depart"],
        station_arrivee=doc["trajet"]["station_arrivee"],
        user_id=doc["utilisateur"]["id_utilisateur"],
        user_nom=doc["utilisateur"]["nom"],
        user_age=doc["utilisateur"]["age"],
        user_genre=doc["utilisateur"]["genre"],
        abonnement=doc["utilisateur"]["abonnement"],
        vehicule_type=doc["vehicule"]["type"],
        vehicule_energie=doc["vehicule"]["energie"],
        temperature=doc["environnement"]["temperature"],
        pluie=doc["environnement"]["pluie"],
        pm10=doc["environnement"]["pm10"],
        volume_trafic=doc["environnement"]["volume_trafic"],
        montant_paiement=doc["paiement"]["montant"],
        moyen_paiement=doc["paiement"]["moyen"],
        device_id=doc["device_id"]
    )
    session.add(row)

session.commit()


In [ ]:
cursor.execute("PRAGMA table_info(event_flat)").fetchall()
# Structure de la table event_flat

[(0, 'event_id', 'INTEGER', 1, None, 1),
 (1, 'date_heure', 'VARCHAR', 0, None, 0),
 (2, 'zone', 'VARCHAR', 0, None, 0),
 (3, 'risque_accident', 'INTEGER', 0, None, 0),
 (4, 'distance_km', 'FLOAT', 0, None, 0),
 (5, 'duree_min', 'FLOAT', 0, None, 0),
 (6, 'ville', 'VARCHAR', 0, None, 0),
 (7, 'station_depart', 'VARCHAR', 0, None, 0),
 (8, 'station_arrivee', 'VARCHAR', 0, None, 0),
 (9, 'user_id', 'INTEGER', 0, None, 0),
 (10, 'user_nom', 'VARCHAR', 0, None, 0),
 (11, 'user_age', 'INTEGER', 0, None, 0),
 (12, 'user_genre', 'VARCHAR', 0, None, 0),
 (13, 'abonnement', 'VARCHAR', 0, None, 0),
 (14, 'vehicule_type', 'VARCHAR', 0, None, 0),
 (15, 'vehicule_energie', 'VARCHAR', 0, None, 0),
 (16, 'temperature', 'FLOAT', 0, None, 0),
 (17, 'pluie', 'INTEGER', 0, None, 0),
 (18, 'pm10', 'FLOAT', 0, None, 0),
 (19, 'volume_trafic', 'INTEGER', 0, None, 0),
 (20, 'montant_paiement', 'FLOAT', 0, None, 0),
 (21, 'moyen_paiement', 'VARCHAR', 0, None, 0),
 (22, 'device_id', 'VARCHAR', 0, None, 0)]

In [140]:
cursor.execute("""
SELECT event_id, zone, ville, user_nom, vehicule_type,
       distance_km, montant_paiement
FROM event_flat
LIMIT 5
""").fetchall()
# Aperçu des données insérées

[(1, 'Sud', 'Paris', 'Lucas', 'Velo', 4.66, 3.93),
 (2, 'Sud', 'Paris', 'Nina', 'Trottinette', 4.24, 1.78),
 (3, 'Nord', 'Paris', 'Lucas', 'Velo', 4.28, 5.31),
 (4, 'Est', 'Paris', 'Hugo', 'Bus', 2.64, 1.35),
 (5, 'Est', 'Paris', 'Emma', 'Bus', 6.91, 3.05)]

Chaque ligne représente un événement complet.

Les colonnes portent des noms explicites,
rendant la table immédiatement exploitable
pour l’analyse ou la visualisation.


##### MIGRATION INVERSE N°2 depuis la dénormalisation « Sécurité par zone »

##### Schéma relationnel cible
```
zone
----
zone_id (PK)
nom

zone_stats
----------
zone_id (FK)
risque_moyen
pollution_moyenne
temperature_moyenne
frequence_pluie
nombre_evenements
alerte
date_calcul


Les indicateurs par zone sont séparés
de la table de référence zone.

Ce modèle facilite les comparaisons temporelles
et les analyses territoriales.


In [128]:
# Implémentation ORM
class Zone(Base):
    __tablename__ = "zone"
    zone_id = Column(Integer, primary_key=True)
    nom = Column(String, unique=True)

class ZoneStats(Base):
    __tablename__ = "zone_stats"
    id = Column(Integer, primary_key=True)
    zone_id = Column(Integer)
    risque_moyen = Column(Float)
    pollution_moyenne = Column(Float)
    temperature_moyenne = Column(Float)
    frequence_pluie = Column(Float)
    nombre_evenements = Column(Integer)
    alerte = Column(String)
    date_calcul = Column(String)


In [130]:
Base.metadata.create_all(engine)

In [131]:
# Insertion ORM
collection = db_mongo["zone_security"]

session.query(ZoneStats).delete()
session.query(Zone).delete()
session.commit()

zones = {}

for doc in collection.find():
    nom = doc["zone"]

    if nom not in zones:
        z = Zone(nom=nom)
        session.add(z)
        session.flush()
        zones[nom] = z

    s = doc["statistiques"]
    row = ZoneStats(
        zone_id=zones[nom].zone_id,
        risque_moyen=s["risque_accident_moyen"],
        pollution_moyenne=s["pollution_pm10_moyenne"],
        temperature_moyenne=s["temperature_moyenne"],
        frequence_pluie=s["frequence_pluie"],
        nombre_evenements=s["nombre_evenements_total"],
        alerte=doc["alerte_securite"],
        date_calcul=doc["derniere_mise_a_jour"]
    )
    session.add(row)

session.commit()


In [141]:
cursor.execute("PRAGMA table_info(zone)").fetchall()
cursor.execute("PRAGMA table_info(zone_stats)").fetchall()
# Structure des tables zone et zone_stats

[(0, 'id', 'INTEGER', 1, None, 1),
 (1, 'zone_id', 'INTEGER', 0, None, 0),
 (2, 'risque_moyen', 'FLOAT', 0, None, 0),
 (3, 'pollution_moyenne', 'FLOAT', 0, None, 0),
 (4, 'temperature_moyenne', 'FLOAT', 0, None, 0),
 (5, 'frequence_pluie', 'FLOAT', 0, None, 0),
 (6, 'nombre_evenements', 'INTEGER', 0, None, 0),
 (7, 'alerte', 'VARCHAR', 0, None, 0),
 (8, 'date_calcul', 'VARCHAR', 0, None, 0)]

La table zone contient les zones géographiques de référence.
La table zone_stats stocke les indicateurs calculés par zone.

Cette séparation permet de conserver
une structure relationnelle claire
tout en facilitant l’analyse.


In [ ]:
cursor.execute("""
SELECT z.nom, zs.risque_moyen, zs.nombre_evenements, zs.alerte
FROM zone_stats zs
JOIN zone z ON z.zone_id = zs.zone_id
LIMIT 5
""").fetchall()
# Aperçu des données insérées

[('Sud', 2.59, 298, 'MOYENNE'),
 ('Nord', 2.49, 292, 'MOYENNE'),
 ('Est', 2.47, 314, 'MOYENNE'),
 ('Centre', 2.56, 286, 'MOYENNE'),
 ('Ouest', 2.45, 310, 'MOYENNE')]

##### MIGRATION INVERSE N°3 depuis « Profil utilisateur marketing »

##### Schéma cible
```
user_profile
------------
user_id (PK)
nom
age
genre
abonnement
nb_trajets
distance_moyenne
duree_moyenne
vehicule_prefere
moyen_paiement_prefere
score_engagement


In [132]:
# Implémentation ORM
class UserProfile(Base):
    __tablename__ = "user_profile"

    user_id = Column(Integer, primary_key=True)
    nom = Column(String)
    age = Column(Integer)
    genre = Column(String)
    abonnement = Column(String)

    nb_trajets = Column(Integer)
    distance_moyenne = Column(Float)
    duree_moyenne = Column(Float)
    vehicule_prefere = Column(String)
    moyen_paiement_prefere = Column(String)


In [134]:
Base.metadata.create_all(engine)

In [135]:
# insertion ORM
collection = db_mongo["user_marketing"]

session.query(UserProfile).delete()
session.commit()

for doc in collection.find():
    row = UserProfile(
        user_id=doc["id_utilisateur"],
        nom=doc["nom"],
        age=doc["age"],
        genre=doc["genre"],
        abonnement=doc["abonnement"],
        nb_trajets=doc["statistiques_usage"]["nombre_trajets"],
        distance_moyenne=doc["statistiques_usage"]["distance_moyenne_km"],
        duree_moyenne=doc["statistiques_usage"]["duree_moyenne_min"],
        vehicule_prefere=doc["preferences"]["vehicule_prefere"],
        moyen_paiement_prefere=doc["preferences"]["moyen_paiement_prefere"]
    )
    session.add(row)

session.commit()


In [143]:
cursor.execute("PRAGMA table_info(user_profile)").fetchall()
# Structure de la table user_profile

[(0, 'user_id', 'INTEGER', 1, None, 1),
 (1, 'nom', 'VARCHAR', 0, None, 0),
 (2, 'age', 'INTEGER', 0, None, 0),
 (3, 'genre', 'VARCHAR', 0, None, 0),
 (4, 'abonnement', 'VARCHAR', 0, None, 0),
 (5, 'nb_trajets', 'INTEGER', 0, None, 0),
 (6, 'distance_moyenne', 'FLOAT', 0, None, 0),
 (7, 'duree_moyenne', 'FLOAT', 0, None, 0),
 (8, 'vehicule_prefere', 'VARCHAR', 0, None, 0),
 (9, 'moyen_paiement_prefere', 'VARCHAR', 0, None, 0)]

La table user_profile regroupe
les informations utilisateur
et les statistiques d’usage calculées.

Elle correspond à une normalisation inverse
orientée CRM et marketing.


In [144]:
cursor.execute("""
SELECT user_id, nom, nb_trajets, distance_moyenne, vehicule_prefere
FROM user_profile
LIMIT 5
""").fetchall()
# Aperçu des données insérées

[(1, 'Alice', 5, 2.59, 'Trottinette'),
 (2, 'Lucas', 5, 6.01, 'Velo'),
 (3, 'Alice', 4, 6.1, 'Bus'),
 (4, 'Lucas', 6, 4.38, 'Voiture'),
 (5, 'Emma', 5, 5.27, 'Bus')]

##### MIGRATION INVERSE N°4 depuis « Analytics global »

##### Schéma cible
```
global_kpi
----------
periode (PK)
nombre_trajets
distance_totale
duree_moyenne
chiffre_affaires
risque_moyen
pollution_moyenne
date_calcul


In [136]:
# Implémentation ORM
class GlobalKPI(Base):
    __tablename__ = "global_kpi"

    periode = Column(String, primary_key=True)
    nombre_trajets = Column(Integer)
    distance_totale = Column(Float)
    duree_moyenne = Column(Float)
    chiffre_affaires = Column(Float)
    risque_moyen = Column(Float)
    pollution_moyenne = Column(Float)
    date_calcul = Column(String)


In [137]:
Base.metadata.create_all(engine)

In [138]:
# Insertion ORM
collection = db_mongo["global_analytics"]

session.query(GlobalKPI).delete()
session.commit()

doc = collection.find_one()

k = doc["kpi_globaux"]

row = GlobalKPI(
    periode=doc["periode"],
    nombre_trajets=k["nombre_trajets"],
    distance_totale=k["distance_totale_km"],
    duree_moyenne=k["duree_moyenne_min"],
    chiffre_affaires=k["chiffre_affaires_total"],
    risque_moyen=k["risque_moyen"],
    pollution_moyenne=k["pollution_moyenne_pm10"],
    date_calcul=doc["derniere_mise_a_jour"]
)

session.add(row)
session.commit()


In [145]:
cursor.execute("PRAGMA table_info(global_kpi)").fetchall()
# Structure de la table global_kpi

[(0, 'periode', 'VARCHAR', 1, None, 1),
 (1, 'nombre_trajets', 'INTEGER', 0, None, 0),
 (2, 'distance_totale', 'FLOAT', 0, None, 0),
 (3, 'duree_moyenne', 'FLOAT', 0, None, 0),
 (4, 'chiffre_affaires', 'FLOAT', 0, None, 0),
 (5, 'risque_moyen', 'FLOAT', 0, None, 0),
 (6, 'pollution_moyenne', 'FLOAT', 0, None, 0),
 (7, 'date_calcul', 'VARCHAR', 0, None, 0)]

La table global_kpi contient
des indicateurs synthétiques globaux.

Elle est conçue pour le pilotage stratégique
et les tableaux de bord décisionnels.


In [146]:
cursor.execute("""
SELECT * FROM global_kpi
""").fetchall()
# Aperçu des données insérées

[('globale',
  1500,
  9314.02,
  13.47,
  5370.23,
  2.51,
  21.12,
  '2026-01-31T18:42:25.460539')]

Les schémas relationnels issus des migrations inverses
sont cohérents avec les dénormalisations NoSQL de départ.

Chaque modèle est adapté à un usage métier précis
et propose une structure lisible et exploitable.

Cette étape valide définitivement
la qualité de nos migrations inverses.


#### COMPARAISON DAO / ORM

Au cours de ce projet, deux approches distinctes
pour accéder aux données relationnelles ont été mises en œuvre :
l’approche DAO basée sur des requêtes SQL explicites
et l’approche ORM basée sur la manipulation d’objets.

Ces deux approches permettent d’interagir
avec une base relationnelle,
mais elles n’apportent pas les mêmes avantages
ni les mêmes contraintes.


##### DAO SQL — Ce que ça apporte

L’approche DAO repose sur l’écriture explicite
de requêtes SQL.

Elle offre un contrôle très précis sur les requêtes,
les jointures et les performances.
Elle permet de comprendre en profondeur
le fonctionnement d’une base relationnelle.

Dans le cadre pédagogique,
le DAO est particulièrement formateur,
car il oblige à raisonner en termes de tables,
de clés étrangères et de jointures.


##### ORM — Ce que ça apporte

L’ORM propose une approche orientée objet
dans laquelle les tables sont représentées
par des classes et les relations par des associations
entre objets.

Cette approche améliore fortement la lisibilité du code
et facilite la manipulation des données.


L’ORM permet de se concentrer sur la logique métier
plutôt que sur l’écriture de requêtes SQL.

Les relations sont gérées automatiquement,
ce qui réduit les risques d’erreurs
liées aux jointures.


Le DAO est plus adapté
lorsqu’un contrôle précis du SQL est nécessaire.

L’ORM est plus adapté
lorsque la priorité est la lisibilité,
la maintenabilité et la cohérence métier.

Ce projet a permis d’explorer de manière concrète
les problématiques de modélisation,
de normalisation et de dénormalisation des données.

À partir d’un fichier CSV brut,
une base relationnelle normalisée a été construite,
puis exploitée pour produire plusieurs modèles NoSQL qui seront à leur tour utilisés
pour produire d'autre bases relationnelles (migration et migration inverse).

Donc Une même donnée peut être représentée
de plusieurs manières différentes
selon les besoins :
- normalisation pour l’intégrité et la cohérence
- dénormalisation pour la performance et la lecture
